# Table of DataFrames

**DataFrame Name** | **Description** | **Based on** | **Exported as**
------- | --------------- | ------------ | -------------
df_orig  | raw DataFrame for hourly energy production | imported condensed hourly energy production data extracted from Transparency Platform |
df  | converted timestamps to datetime and added several derived features | df_orig | 
df_nodst  | undid DST by manipulating column 'dt_date_hour' | df | 
df_hourly  | identical to 'df_nodst' | df_nodst | df_hourly.csv
df_daily  | DataFrame for daily energy production of Germany | df_hourly | df_daily.csv
df_daily_idx | DataFrame using column 'dt_date' as index | df_daily
df_monthly | df_daily grouped by and summed up per month | df_daily |
df_monthly_idx | DataFrame using column 'dt_date' as index | df_monthly |
df_m_all | df_monthly + monthly solar capacity, energy price and weather | df_monthly, df_m_c, df_m_p, df_m_w | df_m_all.csv
df_m_all_hr | Added feature 'mean_sunshine_duration_de_hr' | df_m_all

# Description of columns in DataFrame 'df_hourly'

**Column Number** | **Column Name** | **Data Type** | **Description** | **Renewable Energy?**
------- | --------------- | ------------ | ------------- | ------------
 0  | area                                   | object  | whole Germany |                         
 1  | biomass_mwh                            | float64 | Biomass | renewable                         
 2  | fossil_brown_coal_mwh                  | float64 | Fossil: brown coal | non-renewable                         
 3  | fossil_coal_derived_gas_mwh            | float64 | Fossil: coal-derived gas | non-renewable                          
 4  | fossil_gas_mwh                         | float64 | Fossil: gas | non-renewable                         
 5  | fossil_hard_coal_mwh                   | float64 | Fossil: hard coal | non-renewable                          
 6  | fossil_oil_mwh                         | float64 | Fossil: oil | non-renewable                          
 7  | fossil_oil_shale_mwh                   | float64 | Fossil: oil shale | non-renewable                          
 8  | fossil_peat_mwh                        | float64 | Fossil: peat | non-renewable                          
 9  | geothermal_mwh                         | float64 | Geothermal | renewable                          
 10 | hydro_pumped_storage_aggregated_mwh    | float64 | Hydro pumped storage (generation) | non-renewable                          
 11 | hydro_pumped_storage_consumption_mwh   | float64 | Hydro pumped storage (consumption) |                          
 12 | hydro_run_of_river_and_poundage_mwh    | float64 | Hydro: run-of-river and poundage | renewable                          
 13 | hydro_water_reservoir_mwh              | float64 | Hydro: water reservoir | renewable                         
 14 | marine_mwh                             | float64 | Marine | renewable                          
 15 | nuclear_mwh                            | float64 | Nuclear | non-renewable                          
 16 | other_mwh                              | float64 | Other non-renewables | non-renewable                          
 17 | other_renewable_mwh                    | float64 | Other renewables | renewable                          
 18 | solar_mwh                              | float64 | Solar | renewable                          
 19 | waste_mwh                              | float64 | Waste | non-renewable                          
 20 | wind_offshore_mwh                      | float64 | Wind: offshore | renewable                          
 21 | wind_onshore_mwh                       | float64 | Wind: onshore | renewable
 22 | total_energy_generation_mwh            | float64 | Total energy generation of Germany |                           
 23 | dt_date                                | object  | Date (yyyy-mm-dd) |                          
 24 | dt_date_hour                           | datetime64[ns, pytz.FixedOffset(60)] | Timestamp of 'UTC+1' timezone |
 25 | dt_year                                | int32   | Year (yyyy) |                          
 26 | dt_month                               | int32   | Month of the year (1-12) |                          
 27 | dt_week                                | int32   | Week of the year (1-53) |                           
 28 | dt_dayofweek                           | int32   | Day of the week (0-6) |                           
 29 | dt_dayofmonth                          | int32   | Day of the month (1-31) |                           
 30 | dt_dayofyear                           | int32   | Day of the year (1-366) |                           
 31 | dt_hour                                | int32   | Hour of the day (0-23) |                           
 32 | dt_monthname                           | object  | Name of the month of the year |                           
 33 | dt_dayname                             | object  | Name of the day of the week |                                

**Note: Column 'mtu', 'date', and 'hour' became obsolete after datetime conversion. Since they were NOT updated throughout the DST elimination procedure, these 3 columns were dropped immediately after the procedure.**

# Set up EDA environment

In [ ]:
# Load virtual environment and install all the packages/libraries listed in the file 'requirements.txt' before installing the packages below.

# %pip install xgboost

In [ ]:
# %pip install tqdm

In [ ]:
# %pip install pytz

In [ ]:
# %pip install progressbar2

In [ ]:
# %pip install ydata_profiling

In [ ]:
# %pip install pmdarima==2.0.3

In [ ]:
# Main contents of this notebook:
# 1) EDA of Germany's energy production data extracted from entso-e's transparency platform
# 2) DST handling
# 3) Time series analysis and forecasting based on (S)ARIMA
# 4) Regression modeling of monthly solar energy generation
# 4.1) XGBoost
# 4.2) Ridge
# 5) Forecasting based on regression models
# 5.1) Forecasting with XGBoost model
# 5.2) Forecasting with Ridge linear model

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.dates import MonthLocator
import seaborn as sns
import pytz
from timeit import default_timer as timer
import missingno as msno
import calendar
from calendar import monthrange
from calendar import month_name
from math import sqrt
from scipy.ndimage import gaussian_filter
from tqdm import tqdm
import progressbar
import matplotlib.dates as mdates
from matplotlib.ticker import MaxNLocator, MultipleLocator, FormatStrFormatter, AutoMinorLocator
import matplotlib.ticker as ticker
import itertools
import matplotlib as mpl

from ydata_profiling import ProfileReport

import statsmodels.formula.api as smf
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller,kpss
from statsmodels.tsa.arima_model import ARIMA
from statsmodels.graphics.tsaplots import plot_pacf
import statsmodels.graphics.tsaplots as tsaplot
from statsmodels.tsa.holtwinters import Holt, ExponentialSmoothing, SimpleExpSmoothing
from pmdarima.arima import auto_arima
import pmdarima as pm

from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.linear_model import LogisticRegression, SGDClassifier, Ridge, RidgeCV
from sklearn.metrics import confusion_matrix, accuracy_score, f1_score, fbeta_score, recall_score, precision_score, classification_report, ConfusionMatrixDisplay, make_scorer, mean_absolute_error, mean_squared_error, r2_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.ensemble import AdaBoostClassifier, AdaBoostRegressor, RandomForestClassifier, RandomForestRegressor
from xgboost import XGBClassifier, XGBRegressor
from sklearn.inspection import permutation_importance

# from prophet import Prophet

RSEED = 42

# Ignore warnings
import warnings
warnings.filterwarnings("ignore")

# Display all columns
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)
pd.options.display.max_rows = 100000
# pd.options.display.width = 100000

# Set plotting style
plt.rcParams['font.size'] = 10
plt.rcParams['figure.dpi'] = 150
plt.rcParams['figure.figsize'] = (10, 6)
# Candidate styles: ggplot, fivethirtyeight, solarized, seaborn-paper, paired, set1
# plt.style.use('seaborn-paper')
# style for Seaborn must be one of white, dark, whitegrid, darkgrid, ticks
# sns.set_style('whitegrid')

%matplotlib inline
from pandas.plotting import register_matplotlib_converters
register_matplotlib_converters()

In [ ]:
# Create a new directory for all the outputs

path = '../output'

# Check if the specified path exists or not
is_exist = os.path.exists(path)
if not is_exist:
   os.makedirs(path)    # Create a new directory if it does not already exist

In [ ]:
# Define a custom progress tracker using the tqdm library that provides a progress bar for loops.

def track_progress(results, total_iters):
    pbar = tqdm(total=total_iters, desc="Hyperparameter Tuning")
    for _ in results:
        pbar.update(1)
    pbar.close()

# Overview of the imported data

In [ ]:
df_orig = pd.read_csv('../data/transparency/transparency_20150101-20230716_hourly.csv', low_memory=False)
print(df_orig.columns)
df_orig.head(2)

In [ ]:
# Check the data types and their distributions in the 'Solar_MW' column
data_types_counts = df_orig['solar_mwh'].apply(type).value_counts()

print("Data Types in 'solar_mwh' column and their counts:")
print(data_types_counts)

data_distributions = df_orig['solar_mwh'].value_counts()

In [ ]:
df_orig.info()

In [ ]:
print(df_orig.shape)
print(df_orig.columns)
df_orig.describe().T

# Convert UNIX timestamps and create new features about datetime

In [ ]:
df = df_orig.copy()
df.head(1)

In [ ]:
# Convert UNIX timestamps into pd datetime

df['dt_date'] = pd.to_datetime(df['date'].astype(str), format='%Y%m%d')

df['dt_date_hour'] = pd.to_datetime(df['date'].astype(str) + df['hour'].astype(str), format='%Y%m%d%H')

In [ ]:
# Generate new features about datetime

df['dt_year'] = df['dt_date'].dt.year
df['dt_month'] = df['dt_date'].dt.month
df['dt_week'] = df['dt_date'].dt.isocalendar().week
df['dt_week'] = df['dt_week'].astype('int32')
df['dt_dayofweek'] = df['dt_date'].dt.dayofweek
df['dt_dayofmonth'] = df['dt_date'].dt.day
df['dt_dayofyear'] = df['dt_date'].dt.dayofyear
df['dt_hour'] = df['dt_date_hour'].dt.hour
df['dt_monthname'] = df['dt_date'].dt.month_name()
df['dt_dayname'] = df['dt_date'].dt.day_name()

df.info()
df.head(1)
print(df.shape)
print(df.columns)

In [ ]:
print(df.dt_date_hour.head())
df.head(2)

# Identify duplicates and gaps in 'dt_date_hour' likely caused by daylight saving time each year in Europe

In [ ]:
# Check value_counts for datetime-like features

dt_list = df.filter(regex=r'^dt_', axis=1).columns.to_list()
for item in dt_list:
    print(f'There are {df[item].nunique()} unique values in column {item}.')

In [ ]:
df['hour'].value_counts()

In [ ]:
# Check if there are duplicates in 'dt_date_hour'

print(df.duplicated().sum())

# Print out duplicate items in column 'dt_date_hour' and their occurrence frequencies
# print(df[df.duplicated('dt_date_hour')])

# Get the count of duplicate items in the 'dt_date_hour' column
duplicate_count = df['dt_date_hour'].value_counts()

# Filter and print only the entries with more than one occurrence
print('\n')
print('====='*20)
print("Duplicate items in column 'dt_date_hour' (more than 1 occurrence):")
print(duplicate_count[duplicate_count > 1])
print('====='*20)

In [ ]:
# Examine if there is any discontinuity in the 'dt_date_hour' column caused by the start of daylight saving time in each March in Europe
# Byproduct: a dummy row spotted for year 2015!

# Calculate the time difference between consecutive timestamps
time_diff = df['dt_date_hour'].diff()

# Identify the indices where gaps (discontinuity) occur
gap_indices = time_diff > pd.Timedelta(hours=1)
print(gap_indices[gap_indices == True])

# Print the two ends of each gap (two consecutive entries)
print('\n')
print('====='*20)
print("Two ends (two consecutive entries) of each gap:")
for idx in df.index[gap_indices]:
    print(f"Start: {df['dt_date_hour'][idx-1]}, End: {df['dt_date_hour'][idx]}")
print('====='*20)

**Note: The analysis above makes it clear that the duplicate entries in column 'dt_date_hour' were caused by the end of daylight saving time in (every) October, and the gaps in column 'dt_date_hour' were caused by the start of daylight saving time in (every) March in Europe!**

**Note that in the original data downloaded from the Transparency Platform, there is a dummy row added by somebody to bridge the gap caused by the start of the DST in March of year 2015:
'Germany (DE),20150329 02:00-03:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,20150329,02,0.0'**

**This dummy row needs to be (manually) deleted before one proceeds with DST handling!**

# Locate and delete a dummy row caused by DST of year 2015

In [ ]:
df_nodst = df.copy()
df_nodst.head(2)

## Locate and delete a dummy row inserted at the start of DST of year 2015

In [ ]:
# Locate the dummy row at index=2090 ('mtu' == '20150329 02:00-03:00')

print(df_nodst.iloc[2088:2093])
df_nodst.head(2)

In [ ]:
# Delete the row with the entry '20150329 02:00-03:00' in the 'mtu' column

entry_to_delete = '20150329 02:00-03:00'
df_nodst = df_nodst[df_nodst['mtu'] != entry_to_delete]

# Reset the index
df_nodst = df_nodst.reset_index(drop=True)

## Verify locations of gaps and duplicates after the deletion of the dummy row

In [ ]:
# Verify index continuity after having deleted the dummy row at index=2090 ('mtu' == '20150329 02:00-03:00')

print(df_nodst.iloc[2088:2093])

# Calculate the time difference between consecutive timestamps
time_diff = df_nodst['dt_date_hour'].diff()

# Identify the indices where gaps (discontinuity) occur
gap_indices = time_diff > pd.Timedelta(hours=1)
print(gap_indices[gap_indices == True])

# Print the two ends of each gap (two consecutive entries)
print('\n')
print('====='*20)
print("Two ends (two consecutive entries) of each gap:")
for idx in df_nodst.index[gap_indices]:
    print(f"Start: {df_nodst['dt_date_hour'][idx-1]}, End: {df_nodst['dt_date_hour'][idx]}")
print('====='*20)

# Print the list of dates where gaps exist
dst_start_date_gap_list = []
dst_start_time_gap_list = []
for idx in df_nodst.index[gap_indices]:
    dst_start_date_gap_list.append(df_nodst['dt_date'][idx-1].strftime('%Y-%m-%d'))
    dst_start_time_gap_list.append((df_nodst['dt_date_hour'][idx-1] + pd.Timedelta(hours=1)).strftime('%Y-%m-%d %H:00:00'))
print(dst_start_date_gap_list)
print(dst_start_time_gap_list)

In [ ]:
# Print out duplicate items in the 'dt_date_hour' column

# Calculate the time difference between consecutive timestamps
time_diff = df_nodst['dt_date_hour'].diff()

# Identify the indices where duplicates occur
duplicate_indices = time_diff < pd.Timedelta(hours=1)

# Print the locations of the duplicates in column 'dt_date_hour'
print('\n')
print('====='*20)
print("Date and time of duplicates:")
for idx in df_nodst.index[duplicate_indices]:
    print(f"Duplicate in 'dt_date_hour': {df_nodst['dt_date_hour'][idx-1]}")
print('====='*20)

# Print the list of dates where duplicates exist
dst_end_date_duplicate_list = []
dst_end_time_duplicate_list = []
for idx in df_nodst.index[duplicate_indices]:
    dst_end_date_duplicate_list.append(df_nodst['dt_date'][idx-1].strftime('%Y-%m-%d'))
    dst_end_time_duplicate_list.append((df_nodst['dt_date_hour'][idx-1]).strftime('%Y-%m-%d %H:00:00'))    
print(dst_end_date_duplicate_list)
print(dst_end_time_duplicate_list)

# Tackle DST

## Tackle DST by eliminating gaps and duplicates in column 'dt_date_hour'

In [ ]:
# Eliminate all the gaps and duplicates (caused by DST) in column 'dt_date_hour' by using the 'pytz' library with 'UTC+1' (CET) as the time zone

# Create a timezone object for 'UTC+1' (CET) using pytz
timezone_cet = pytz.FixedOffset(60)  # UTC+1 in minutes

# Convert the 'dt_date_hour' column to a timezone-aware datetime format (using the CET timezone)
df_nodst['dt_date_hour'] = pd.to_datetime(df_nodst['dt_date_hour']).dt.tz_localize(timezone_cet)

# Lists containing exact DST starting and end dates for years from 2015 till 2023
dst_start_dates = ['2015-03-29 02:00:00', '2016-03-27 02:00:00', '2017-03-26 02:00:00', '2018-03-25 02:00:00', '2019-03-31 02:00:00', '2020-03-29 02:00:00', '2021-03-28 02:00:00', '2022-03-27 02:00:00', '2023-03-26 02:00:00']
dst_end_dates = ['2015-10-25 02:00:00', '2016-10-30 02:00:00', '2017-10-29 02:00:00', '2018-10-28 02:00:00', '2019-10-27 02:00:00', '2020-10-25 02:00:00', '2021-10-31 02:00:00', '2022-10-30 02:00:00', '2023-10-29 02:00:00']

# Adjust the 'dt_date_hour' column to eliminate gaps and duplicates caused by DST
for start_date, end_date in zip(dst_start_dates, dst_end_dates):
    start_date = pd.to_datetime(start_date).tz_localize(timezone_cet)
    end_date = pd.to_datetime(end_date).tz_localize(timezone_cet)

    # Find rows that need to be adjusted based on DST changes
    mask = (df_nodst['dt_date_hour'] >= start_date) & (df_nodst['dt_date_hour'] <= end_date) & ~df_nodst['dt_date_hour'].duplicated()
    adjusted_hours = df_nodst[mask]['dt_date_hour'] - pd.Timedelta(hours=1)

    # Update the original DataFrame with the adjusted values
    df_nodst.loc[mask, 'dt_date_hour'] = adjusted_hours

# Sort the DataFrame by 'dt_date_hour' to ensure it is in the correct order
df_nodst = df_nodst.sort_values(by='dt_date_hour')

# Drop duplicate entries (if any)
# df_nodst = df_nodst.drop_duplicates(subset='dt_date_hour', keep='first')

# Reset the index after removing duplicates (if needed)
df_nodst = df_nodst.reset_index(drop=True)

print(df_nodst.shape)


## Verify timestamp continuity after handling DST

In [ ]:
# Verify index continuity after DST adjustment

print(df_nodst.iloc[2088:2093])

# Calculate the time difference between consecutive timestamps
time_diff = df_nodst['dt_date_hour'].diff()

# Identify the indices where gaps (discontinuity) occur
gap_indices = time_diff > pd.Timedelta(hours=1)
print(gap_indices[gap_indices == True])

# Print the two ends of each gap (two consecutive entries)
print('\n')
print('====='*20)
print("Two ends (two consecutive entries) of each gap:")
for idx in df_nodst.index[gap_indices]:
    print(f"Start: {df_nodst['dt_date_hour'][idx-1]}, End: {df_nodst['dt_date_hour'][idx]}")
print('====='*20)

# Print the list of dates where gaps exist
dst_start_date_gap_list = []
dst_start_time_gap_list = []
for idx in df_nodst.index[gap_indices]:
    dst_start_date_gap_list.append(df_nodst['dt_date'][idx-1].strftime('%Y-%m-%d'))
    dst_start_time_gap_list.append((df_nodst['dt_date_hour'][idx-1] + pd.Timedelta(hours=1)).strftime('%Y-%m-%d %H:00:00'))
print(dst_start_date_gap_list)
print(dst_start_time_gap_list)

In [ ]:
# Check if there are still duplicate items in the 'dt_date_hour' column after DST adjustment

# Calculate the time difference between consecutive timestamps
time_diff = df_nodst['dt_date_hour'].diff()

# Identify the indices where duplicates occur
duplicate_indices = time_diff < pd.Timedelta(hours=1)

# Print the locations of the duplicates in column 'dt_date_hour'
print('\n')
print('====='*20)
print("Date and time of duplicates:")
for idx in df_nodst.index[duplicate_indices]:
    print(f"Duplicate in 'dt_date_hour': {df_nodst['dt_date_hour'][idx-1]}")
print('====='*20)

# Print the list of dates where duplicates exist
dst_end_date_duplicate_list = []
dst_end_time_duplicate_list = []
for idx in df_nodst.index[duplicate_indices]:
    dst_end_date_duplicate_list.append(df_nodst['dt_date'][idx-1].strftime('%Y-%m-%d'))
    dst_end_time_duplicate_list.append((df_nodst['dt_date_hour'][idx-1]).strftime('%Y-%m-%d %H:00:00'))    
print(dst_end_date_duplicate_list)
print(dst_end_time_duplicate_list)

In [ ]:
df_nodst.head(2)

In [ ]:
df_nodst.tail(2)

In [ ]:
df_nodst.info()

## Update datetime-like features after handling DST 

In [ ]:
# Update new features about datetime based on column 'dt_date_time' after DST adjustment

df_nodst['dt_year'] = df_nodst['dt_date_hour'].dt.year
df_nodst['dt_month'] = df_nodst['dt_date_hour'].dt.month
df_nodst['dt_week'] = df_nodst['dt_date_hour'].dt.isocalendar().week
df_nodst['dt_week'] = df_nodst['dt_week'].astype('int32')
df_nodst['dt_dayofweek'] = df_nodst['dt_date_hour'].dt.dayofweek
df_nodst['dt_dayofmonth'] = df_nodst['dt_date_hour'].dt.day
df_nodst['dt_dayofyear'] = df_nodst['dt_date_hour'].dt.dayofyear
df_nodst['dt_hour'] = df_nodst['dt_date_hour'].dt.hour
df_nodst['dt_monthname'] = df_nodst['dt_date_hour'].dt.month_name()
df_nodst['dt_dayname'] = df_nodst['dt_date_hour'].dt.day_name()
df_nodst['dt_date'] = df_nodst['dt_date_hour'].dt.date

In [ ]:
print(df_nodst.shape)
print(df_nodst.columns)
print(df_nodst.dt_date.head())
df_nodst.head(2)

## Drop low-value columns and rows after handling DST

In [ ]:
# Drop columns that have little value for further EDA

df_nodst = df_nodst.drop(['mtu', 'date', 'hour'], axis=1)

In [ ]:
# Drop rows associated with the last incomplete day

# Drop rows where 'dt_date' is '2023-07-17'

# Convert the filter value to datetime.date object
target_date = pd.to_datetime('2023-07-17').date()

# Drop rows where 'dt_date' is '2023-07-17'
df_nodst = df_nodst[df_nodst['dt_date'] != target_date]

# Reset the index after dropping rows
df_nodst = df_nodst.reset_index(drop=True)


In [ ]:
df_nodst.info()

In [ ]:
dt_list = df_nodst.filter(regex=r'^dt_', axis=1).columns.to_list()
for item in dt_list:
    print(f'There are {df_nodst[item].nunique()} unique values in column {item}.')

# Play around a bit after handling DST

In [ ]:
print(f"Number of hours with 0 solar energy generation:")
df_nodst.query('solar_mwh == 0.0').groupby(['dt_year'])['solar_mwh'].count()

In [ ]:
print(f"Breakdown [%] of solar energy generation based on hours of the day:")
total_solar_generation = df_nodst['solar_mwh'].sum()
df_nodst.groupby(['dt_hour'])['solar_mwh'].sum()/total_solar_generation*100

In [ ]:
print(f"Breakdown [%] of solar energy generation based on months of the year:")
total_solar_generation = df_nodst['solar_mwh'].sum()
df_nodst.groupby(['dt_month'])['solar_mwh'].sum()/total_solar_generation*100

In [ ]:
print(f"Breakdown [%] of solar energy generation based on days of a week:")
total_solar_generation = df_nodst['solar_mwh'].sum()
df_nodst.groupby(['dt_dayofweek'])['solar_mwh'].sum()/total_solar_generation*100

In [ ]:
print(f"Yearly solar energy generation [MWh]:")
total_solar_generation = df_nodst['solar_mwh'].sum()
df_nodst.groupby(['dt_year'])['solar_mwh'].sum()

# BONUS: A versatile function that combines 'info' and 'describe'

In [ ]:
# Poor man's approach to show description of features of various data types

# Get basic statistics for non-numeric datatypes
describe_non_numeric = df_nodst.describe(include=['datetime','object']).T

print(describe_non_numeric)
print(df_nodst.shape)

# Convert column 'dt_date_hour' to a string representation without the timezone
df_nodst['dt_date_hour_str'] = df_nodst['dt_date_hour'].dt.tz_localize(None).dt.strftime('%Y-%m-%d %H:%M:%S')

# Now one can use the 'describe' method with 'include' parameter to describe the datetime column
describe_datetime = df_nodst['dt_date_hour_str'].describe()

# Optionally, one can drop the intermediate column 'dt_date_hour_str'
df_nodst.drop(columns=['dt_date_hour_str'], inplace=True)

print(describe_datetime)

In [ ]:
from IPython.display import display

def info_describe(df, N_MOST_FREQ=5):
    
    # Display top-N_MOST_FREQ most frequent unique values

    # Get basic statistics for numeric columns using df.describe()
    describe_numeric = df.describe().transpose()

    # Combine the information into a single DataFrame
    feature_info_list = []

    for i, col in enumerate(df.columns):
        
        data_type = df[col].dtype
        non_null_count = df[col].count()
        unique_count = df[col].nunique()
        top = df[col].mode().iloc[0] if non_null_count > 0 else None
        frequency = df[col].value_counts().to_dict() if non_null_count > 0 else None
        if data_type != 'object' and data_type != 'datetime64[ns]' and data_type != 'datetime64[ns, pytz.FixedOffset(60)]':  # Numeric columns
            describe_numeric_values = describe_numeric.loc[col].tolist()
        else:
            describe_numeric_values = [None] * 8  # Placeholder values for non-numeric columns (number of columns from output of df.describe().transpose())

        if frequency is not None and len(frequency) > N_MOST_FREQ:  # Limit to top-N_MOST_FREQ most frequent unique values
            top_n_freq = dict(list(frequency.items())[:N_MOST_FREQ])
            frequency_str = str(top_n_freq)[:-1] + ', ...}'
        else:
            frequency_str = str(frequency)

        feature_info = {
            # 'Feature Index': i,
            'Feature Name': col,
            'Data Type': data_type,
            'Non-Null Count': non_null_count,
            'Num of Unique Values': unique_count,
            'Mean': describe_numeric_values[1],
            'Std': describe_numeric_values[2],
            'Min': describe_numeric_values[3],
            '25%': describe_numeric_values[4],
            '50%': describe_numeric_values[5],
            '75%': describe_numeric_values[6],
            'Max': describe_numeric_values[7],
            'Most Frequent Value': top,
            'Most Frequent Unique Values': frequency_str
        }
        feature_info_list.append(feature_info)

    # Create the final DataFrame
    feature_info_df = pd.DataFrame(feature_info_list)

    # Print the final DataFrame using display()
    print("\nDataFrame Description and Unique Value Counts:")

    # No wrapping for the display of the final DataFrame
    feature_info_df_nowrap = feature_info_df.style.set_table_styles([dict(selector="td", props=[('white-space', 'nowrap')])])
    display(feature_info_df_nowrap)
    

In [ ]:
# Alternative way of disabling word-wrap when displaying a DataFrame

# %%html
# <style>
# .dataframe td {
#     white-space: nowrap;
# }
# </style>


In [ ]:
df_hourly = df_nodst.copy()
info_describe(df_hourly, 5)

In [ ]:
# Create the box plot for daily solar generation using Seaborn

plt.figure(figsize=(6, 4))
sns.boxplot(x='dt_hour', y='solar_mwh', data=df_hourly)
plt.title('Distribution of hourly solar energy generation of Germany (2015 - 2023)')
plt.xlabel('Hour of the day')
plt.ylabel('Hourly solar energy generation [MWh]')
plt.show()

# Condense 'df_hourly' into 'df_daily'

## Export 'df_hourly'

In [ ]:
# %pip install --upgrade ydata-profiling

In [ ]:
# %pip install --upgrade openpyxl

In [ ]:
# Export 'df_hourly' to CSV
# df_hourly.to_csv('../output/df_hourly.csv', sep=',', index=False)

# # Export the first 10000 rows of 'df_hourly' to Excel
# df_hourly.head(10000).to_excel('../output/df_hourly.xlsx')

# # Generate profile report using ProfileReport
# df_hourly_profile = ProfileReport(df_hourly, title="Profiling Report")
# # Export profile report
# df_hourly_profile.to_file("../output/df_hourly_profile.html")

In [ ]:
print(df_nodst.columns)

In [ ]:
energy_types = ['biomass_mwh', 'fossil_brown_coal_mwh',
       'fossil_coal_derived_gas_mwh', 'fossil_gas_mwh', 'fossil_hard_coal_mwh',
       'fossil_oil_mwh', 'fossil_oil_shale_mwh', 'fossil_peat_mwh',
       'geothermal_mwh', 'hydro_pumped_storage_aggregated_mwh',
       'hydro_pumped_storage_consumption_mwh',
       'hydro_run_of_river_and_poundage_mwh', 'hydro_water_reservoir_mwh',
       'marine_mwh', 'nuclear_mwh', 'other_mwh', 'other_renewable_mwh',
       'solar_mwh', 'waste_mwh', 'wind_offshore_mwh', 'wind_onshore_mwh',
       'total_energy_generation_mwh']

renewable_energy_types = ['biomass_mwh', 'geothermal_mwh',
       'hydro_run_of_river_and_poundage_mwh', 'hydro_water_reservoir_mwh',
       'marine_mwh', 'other_renewable_mwh',
       'solar_mwh', 'wind_offshore_mwh', 'wind_onshore_mwh']

nonrenewable_energy_types = ['fossil_brown_coal_mwh',
       'fossil_coal_derived_gas_mwh', 'fossil_gas_mwh', 'fossil_hard_coal_mwh',
       'fossil_oil_mwh', 'fossil_oil_shale_mwh', 'fossil_peat_mwh',
       'hydro_pumped_storage_aggregated_mwh',
       'nuclear_mwh', 'other_mwh', 'waste_mwh']

## Create 'df_daily' for daily energy generation of Germany

In [ ]:
# Create a new DataFrame for daily energy generation

df_temp = df_nodst.copy()

daily_columns = ['dt_date_hour'] + energy_types
df_daily = df_temp[daily_columns].groupby(pd.Grouper(key='dt_date_hour', freq='D')).sum().reset_index()

In [ ]:
print(df_daily.shape)
df_daily.head()

In [ ]:
# Create new features about datetime based on column 'dt_date_hour'

df_daily['dt_year'] = df_daily['dt_date_hour'].dt.year
df_daily['dt_month'] = df_daily['dt_date_hour'].dt.month
df_daily['dt_week'] = df_daily['dt_date_hour'].dt.isocalendar().week
df_daily['dt_week'] = df_daily['dt_week'].astype('int32')
df_daily['dt_dayofweek'] = df_daily['dt_date_hour'].dt.dayofweek
df_daily['dt_dayofmonth'] = df_daily['dt_date_hour'].dt.day
df_daily['dt_dayofyear'] = df_daily['dt_date_hour'].dt.dayofyear
# df_daily['dt_hour'] = df_daily['dt_date_hour'].dt.hour
df_daily['dt_monthname'] = df_daily['dt_date_hour'].dt.month_name()
df_daily['dt_dayname'] = df_daily['dt_date_hour'].dt.day_name()
df_daily['dt_date'] = df_daily['dt_date_hour'].dt.date

In [ ]:
info_describe(df_daily, 2)

## A simple yet informative plot :)

In [ ]:
# Note: Seaborn 0.12.2, no need for preset plotting style

# Pivot 'df_daily' to have years as rows and days of the year as columns

pivot_table = df_daily.pivot_table(index='dt_year', columns='dt_dayofyear', values='solar_mwh', aggfunc='sum')

# Divide values by 1000 so that unit becomes 'GWh'
pivot_table /= 1000.0

# Draw a heatmap using Seaborn for daily solar energy production from 2015-01-01 till 2023-07-16
plt.figure(figsize=(10, 5))
ax = sns.heatmap(pivot_table, cmap='RdYlBu_r', linewidths=0.0, linecolor='black', cbar_kws={'label': 'Daily solar energy generation [GWh]', 'pad': 0.03})
plt.title("Germany's daily solar energy generation (2015 - 2023)")
plt.xlabel('Day of the year')
plt.ylabel('Year')
ax.set_xticks([])  # Remove x-axis ticks and labels
ax.set_xticks(range(15, 366, 30), minor=True)  # Add minor ticks for month boundaries
ax.set_xticklabels(pd.date_range('2023-01-01', '2023-12-31', freq='M').strftime('%b'), minor=True)  # Set month names for minor ticks

plt.grid(False)
# Alternatively
# ax.grid(False)

plt.show()

# Basic TSA for 'df_daily' and 'df_daily_idx'

## Data visualization

In [ ]:
# Hourly solar energy production in Germany

fig, ax = plt.subplots()
sns.lineplot(x='dt_date_hour', y='solar_mwh', data=df_hourly, ax=ax, linewidth=0.1)
ax.set(title="Germany's hourly solar energy generation", xlabel='Timestamp', ylabel='Solar energy generation [MWh]')

plt.figure(figsize=(15, 5))
plt.show()

In [ ]:
df_daily['solar_gwh'] = df_daily['solar_mwh'] / 1000.0
df_daily['total_energy_generation_gwh'] = df_daily['total_energy_generation_mwh'] / 1000.0
info_describe(df_daily, 2)

In [ ]:
# Daily solar energy production in Germany

fig, ax = plt.subplots()
sns.lineplot(x='dt_date', y='solar_gwh', data=df_daily, ax=ax, linewidth=0.25)
ax.set(title="Germany's daily solar energy generation", xlabel='Timestamp', ylabel='Solar energy generation [GWh]')

plt.figure(figsize=(12, 5))
plt.show()

In [ ]:
# Plot yearly seasonality

fig, ax = plt.subplots() 

pd.pivot_table(data=df_daily[['dt_year', 'dt_dayofyear', 'solar_gwh']], index='dt_dayofyear', columns='dt_year')['solar_gwh'].plot(cmap='viridis', alpha=0.5, ax=ax)

ax.legend(title='year', loc='center left', bbox_to_anchor=(1, 0.5))
ax.set(title="Germany's daily solar energy generation", xlabel='Day of the year', ylabel='Solar energy generation [GWh]')

plt.figure(figsize=(12, 5))
plt.show()

In [ ]:
# Polar plot for seasonality 

ax = plt.subplot(111, projection='polar')

# Convert and plot data
df_daily.assign(day_of_year_cyclic = lambda x: x['dt_dayofyear'].transform(lambda x: 2*np.pi*x/365.25)) \
    .pipe((sns.lineplot, 'data'), 
        x='day_of_year_cyclic', 
        y='solar_gwh', 
        hue='dt_month',
        palette=sns.color_palette("husl", 14),
        ax=ax
    )

fancy_plot=True     # Make the plot more beautiful

if(fancy_plot): 
    # Find out how many days each month has
    days_per_month=[0] + [monthrange(2022, i)[1] for i in range(1,12)]  
    # Get each month's starting day
    month_start=np.cumsum(days_per_month) + 1
    # Convert starting day into an angle (in rad) by using 365.25 as the average length of a year
    month_start_theta=[i * 2 * np.pi / 365.25 for i in month_start]          

    month_label=[month_name[i] for i in range(1,13)]
    month_label_long=[label + '\n(Day ' + str(month_start[ind]) + ')' for ind,label in enumerate(month_label)]

    ax.set_title("Germany's daily solar energy generation", va='bottom', pad=22);
    ax.spines.clear()
    
    ax.set_xlabel('')
    ax.set_xticks(month_start_theta)
    ax.set_xticklabels(month_label_long)
    
    ax.set_ylabel('')    
    ax.set_ylim(-5,300)
    ax.set_yticks(yt:=[0,100,200])
    ax.set_yticklabels([str(t)+'GWh' for t in yt], rotation = 45)

    # Arrows and Annotations
    style = "Simple, tail_width=0.5, head_width=4, head_length=8"
    kw = dict(arrowstyle=style, color="black")
    ax.set_rlabel_position(1) 
    ax.text(13*2*np.pi/360,250,"Days",size=10,color='black',rotation=-80,va='center')
    ax.text(-3*2*np.pi/360,150,"Energy Generation",size=10,color='black',rotation=-0,va='center')

    # Horizontal arrow
    a1 = patches.FancyArrowPatch((1*np.pi/180, -5), (1*np.pi/180, 280), **kw)
    # Rotational arrow
    a2 = patches.FancyArrowPatch((1*np.pi/180, 250), (25*np.pi/180, 250),
                                connectionstyle=f"arc3,rad={0.105}", **kw)
    
    ax.add_patch(a1)
    ax.add_patch(a2)

    ax.set_rorigin(-5)
    ax.xaxis.set_tick_params(which='major',pad=10)

    ax.legend(labels=month_label,ncol=2,facecolor='white',edgecolor='white',bbox_to_anchor=(1.1, 1.1), loc=1)

    ax.figure.set_figwidth(10)
    ax.figure.set_figheight(10)

## Time series decomposition for 'df_daily_idx'

In [ ]:
# Customize a palette

NF_ORANGE = '#ff5a36'
NF_BLUE = '#163251'
cmaps_hex = ['#193251','#FF5A36','#696969', '#7589A2','#FF5A36', '#DB6668']
sns.set_palette(palette=cmaps_hex)
sns_c = sns.color_palette(palette=cmaps_hex)

In [ ]:
sns_c

In [ ]:
# Set 'dt_date' as the index of the DataFrame 'df_daily_idx'
df_daily_idx = df_daily.set_index('dt_date', inplace=False)
df_daily_idx.head()

In [ ]:
# Decompose df_daily_idx using 'seasonal_decompose'

# Use the parameter 'period=365' to extract the yearly seasonality

decomposition = seasonal_decompose(x=df_daily_idx['solar_gwh'], 
                                   model='multiplicative',
                                   two_sided=True,
                                   period=365)


fig, ax = plt.subplots(4, 1, figsize=(12, 12), constrained_layout=True)
decomposition.observed.plot(c=sns_c[0], ax=ax[0])
ax[0].set(title="Germany's daily solar energy generation [GWh]",xlabel='Date')
decomposition.trend.plot(c=sns_c[1], ax=ax[1])
ax[1].set(title='Trend component',xlabel='Date')
decomposition.seasonal.plot(c=sns_c[2], ax=ax[2])
ax[2].set(title='Seasonal component',xlabel='Date')
decomposition.resid.plot(c=sns_c[3], ax=ax[3])
ax[3].set(title='Residual component',xlabel='Date')
fig.set_size_inches(20, 10)



## Check stationarity

In [ ]:
# Test for stationarity 

def stationarity_test(energy_generation):
    
    # Calculate rolling mean and rolling standard deviation
    rolling_mean = energy_generation.rolling(365).mean()
    rolling_std_dev = energy_generation.rolling(365).std()
    
    # Plot the statistics
    plt.figure(figsize=(24,6))
    plt.plot(rolling_mean, color='#FF5A36', label='Rolling Mean')
    plt.plot(rolling_std_dev, color='#1E4485', label = 'Rolling Std Dev')
    plt.plot(energy_generation, color='#99D04A',label='Original Time Series')
    # plt.xticks([])
    plt.legend(loc='best')
    plt.title('Rolling Mean and Standard Deviation')
    
    # ADF test
    print("ADF Test:")
    adf_test = adfuller(energy_generation,autolag='AIC')
    print('Null Hypothesis: Not Stationary')
    print('ADF Statistic: %f' % adf_test[0])
    print('p-value: %f' % adf_test[1])
    print('----'*10)
    
    # KPSS test
    print("KPSS Test:")
    kpss_test = kpss(energy_generation, regression='c', nlags="legacy", store=False)
    print('Null Hypothesis: Stationary')
    print('KPSS Statistic: %f' % kpss_test[0])
    print('p-value: %f' % kpss_test[1])
    print('----'*10)
    
stationarity_test(df_daily_idx['solar_gwh'])

## De-trend the time series

In [ ]:
# De-trending the time series
df_daily_idx['solar_gwh_detrend'] = (df_daily_idx['solar_gwh'] - df_daily_idx['solar_gwh'].shift(365))

In [ ]:
# Test for stationarity after de-trending 

def stationarity_test_dt(energy_generation):
    
    # Calculate rolling mean and rolling standard deviation
    rolling_mean = energy_generation.rolling(365).mean()
    rolling_std_dev = energy_generation.rolling(365).std()
  
    # Plot the statistics
    plt.figure(figsize=(24,6))
    plt.plot(rolling_mean, label='Rolling Mean',linewidth=2.0)
    plt.plot(rolling_std_dev, label = 'Rolling Std Dev',linewidth=2.0)
    plt.plot(energy_generation,label='De-Trended Time Series')
    # plt.xticks([])
    plt.legend(loc='best')
    plt.title('Rolling Mean and Standard Deviation')
    plt.tight_layout()
    
    # ADF test
    print("ADF Test:")
    adf_test = adfuller(energy_generation,autolag='AIC')
    print('Null Hypothesis: Not Stationary')
    print('ADF Statistic: %f' % adf_test[0])
    print('p-value: %f' % adf_test[1])
    print('----'*10)
    
    # KPSS test
    print("KPSS Test:")
    kpss_test = kpss(energy_generation, regression='c', nlags='legacy', store=False)
    print('Null Hypothesis: Stationary')
    print('KPSS Statistic: %f' % kpss_test[0])
    print('p-value: %f' % kpss_test[1])
    print('----'*10)
    
stationarity_test_dt(df_daily_idx['solar_gwh_detrend'].dropna())

# Partial Autocorrelation Plot
pacf = plot_pacf(df_daily_idx['solar_gwh_detrend'].dropna(), lags=365)

## Data splitting

In [ ]:
df_daily_idx.head(2)

In [ ]:
print(df_daily_idx.shape[0])
print(df_daily_idx.query('dt_year == 2023')['dt_year'].count())
df_daily_idx['dt_year'].value_counts()

In [ ]:
# Split data into train and test set

totol_rows = df_daily_idx.shape[0]
year_2023_rows = df_daily_idx.query('dt_year == 2023')['dt_year'].count()
split_ratio = (totol_rows - year_2023_rows) * 1.0 / totol_rows
print(f"Split ratio: {split_ratio}")

df_arima = df_daily_idx['solar_gwh']
train_test_split_ratio = int(len(df_arima)*split_ratio)
train_data, test_data = df_arima[:train_test_split_ratio], df_arima[train_test_split_ratio:]

# Plotting the train and test set
plt.figure(figsize=(10,6))
plt.xlabel('Year')
plt.ylabel('Daily solar energy generation [GWh]')
# plt.xticks([])
plt.plot(train_data, 'red', label='Train data')
plt.plot(test_data, 'black', label='Test data')
plt.legend()

# ARIMA (Auto-Regressive Integrated Moving Average)

## Auto ARIMA without de-trend

In [ ]:
# Auto ARIMA Method
arima_model = auto_arima(train_data,
                      start_p=0, start_q=0,
                      max_p=5, max_q=5,
                      test='adf',        
                      trace=True,
                      alpha=0.05,
                      scoring='mse',
                      suppress_warnings=True,
                      seasonal = False
                      )

# Fit the final model with the order
fitted_model = arima_model.fit(train_data) 
print(fitted_model.summary())

# Forecasting values
forecast_values = fitted_model.predict(len(test_data), alpha=0.05) 
# fcv_series = pd.Series(forecast_values[0], index=test_data.index)
fcv_series = forecast_values

# Plot the predicted stock price and original price
plt.figure(figsize=(12,5), dpi=100)
plt.plot(train_data, label='training')
plt.plot(test_data, label='Actual solar energy generation')
plt.plot(fcv_series,label='Predicted solar energy generation')
plt.title('Prediction for daily solar energy generation')
plt.xlabel('Time')
plt.ylabel('Daily solar energy generation [GWh]')
# plt.xticks([])
plt.legend(loc='upper left', fontsize=8)
#plt.show()

# Evaluate the model by calculating RMSE
rms_auto_arima = sqrt(mean_squared_error(test_data.values, fcv_series))
print("Auto-Arima RMSE :- " + str(round(rms_auto_arima,3)))

## Auto ARIMA with de-trend

In [ ]:
# Dropna for 'solar_gwh_detrend'

print(df_daily_idx['solar_gwh_detrend'].head(2))
print(df_daily_idx['solar_gwh_detrend'].tail(2))

s1 = df_daily_idx['solar_gwh_detrend'].dropna()

print(s1.shape)

### Data splitting

In [ ]:
# Split data into train and test set

totol_rows = s1.shape
year_2023_rows = df_daily_idx.query('dt_year == 2023')['dt_year'].count()
split_ratio = (totol_rows - year_2023_rows) * 1.0 / totol_rows
print(f"Split ratio: {split_ratio}")

df_arima_detr = s1
train_test_split_ratio = int(len(df_arima_detr)*split_ratio)
train_data_detr, test_data_detr = df_arima_detr[:train_test_split_ratio], df_arima_detr[train_test_split_ratio:]

# Plotting the train and test set
plt.figure(figsize=(10,6))
plt.xlabel('Year (after de-trending)')
plt.ylabel('Daily solar energy generation (after de-trending)')
# plt.xticks([])
plt.plot(train_data_detr, 'red', label='Train data')
plt.plot(test_data_detr, 'black', label='Test data')
plt.legend()

### Simple 'auto_arima'

In [ ]:
# Auto ARIMA Method
arima_model = auto_arima(train_data_detr,
                      start_p=0, start_q=0,
                      max_p=5, max_q=5,
                      test='adf',        
                      trace=True,
                      alpha=0.05,
                      scoring='mse',
                      suppress_warnings=True,
                      seasonal = False
                      )

# Fit the final model with the order
fitted_model = arima_model.fit(train_data_detr) 
print(fitted_model.summary())

# # Forecasting values
# forecast_values = fitted_model.predict(len(test_data_detr), alpha=0.05) 
# # fcv_series = pd.Series(forecast_values[0], index=test_data_detr.index)

# Step 1: Make Forecasts
n_forecast_steps = len(test_data_detr)
forecasts, conf_int = fitted_model.predict(n_periods=n_forecast_steps, return_conf_int=True)

# Step 2: Inverse Transformation
# Calculate the cumulative sum of the original time series data from the last observed value
cumulative_sum = s1.iloc[-1]
inverse_forecasts = cumulative_sum + forecasts.cumsum()

# Inverse-transform confidence intervals
lower_bound = cumulative_sum + conf_int[:, 0].cumsum()
upper_bound = cumulative_sum + conf_int[:, 1].cumsum()

# Create a DataFrame to store the forecasts and confidence intervals (if available)
forecast_df = pd.DataFrame({
    'Forecast': inverse_forecasts,
    'Lower Bound': lower_bound,
    'Upper Bound': upper_bound
})

fcv_series = forecast_df['Forecast']

# Plot the predicted stock price and original price
plt.figure(figsize=(12,5), dpi=100)
plt.plot(train_data, label='training')
plt.plot(test_data, label='Actual solar energy generation')
plt.plot(fcv_series,label='Predicted solar energy generation')
plt.title('Prediction for daily solar (with de-trending)')
plt.xlabel('Time')
plt.ylabel('Daily solar energy generation [GWh]')
# plt.xticks([])
plt.legend(loc='upper left', fontsize=8)
#plt.show()

# Evaluate the model by calculating RMSE
rms_auto_arima = sqrt(mean_squared_error(test_data_detr.values, fcv_series))
print("Auto-Arima RMSE :- " + str(round(rms_auto_arima,3)))

### ARIMA with rolling-forecast

#### 'df_daily'

In [ ]:
print(f"Original train data: {train_data.index[0]} - {train_data.index[-1]}")
print(f"Original test data: {test_data.index[0]} - {test_data.index[-1]}")

print(f"De-trended train data: {train_data_detr.index[0]} - {train_data_detr.index[-1]}")
print(f"De-trended test data: {test_data_detr.index[0]} - {test_data_detr.index[-1]}")

train_data_length = len(train_data) 
train_data_detr_length = len(train_data_detr)

print(train_data_length, test_data.shape, train_data_detr_length, test_data_detr.shape)


In [ ]:
# Shorten training data

to_ditch_train_rows = df_daily_idx.query('dt_year==2015 or dt_year==2016 or dt_year==2017 or dt_year==2018')['dt_year'].count()

train_data_shortened = train_data[to_ditch_train_rows:]
train_data_detr_shortened = train_data_detr[(train_data_detr_length-(train_data_length-to_ditch_train_rows)):]

print(f"Shortened original train data: {train_data_shortened.index[0]} - {train_data_shortened.index[-1]}")
print(f"Original test data: {test_data.index[0]} - {test_data.index[-1]}")

print(f"Shortened de-trended train data: {train_data_detr_shortened.index[0]} - {train_data_detr_shortened.index[-1]}")
print(f"De-trended test data: {test_data_detr.index[0]} - {test_data_detr.index[-1]}")


In [ ]:
# ARIMA with rolling-forecast

import pmdarima as pm

arima_model = auto_arima(train_data_detr_shortened,
                      start_p=0, start_q=0,
                      max_p=1, max_q=1,
                      test='adf',        
                      trace=True,
                      alpha=0.05,
                      scoring='mse',
                      suppress_warnings=True,
                      seasonal=True,
                      # m=365,  # requires too much computation time!
                      n_jobs=-1,
                      stepwise=False
                      )

# Fit the final model with the order
fitted_model = arima_model.fit(train_data_detr_shortened) 
print(fitted_model.summary())

# Define the rolling window size (e.g., 365 days)
window_size = 365
n_forecast_steps = len(test_data_detr)

# Initialize lists to store forecasts and actual values
forecasts = []
actual_values = []

# Obtain ARIMA model orders from the fitted model
p = fitted_model.order[0]
d = fitted_model.order[1]
q = fitted_model.order[2]
P = fitted_model.seasonal_order[0]
D = fitted_model.seasonal_order[1]
Q = fitted_model.seasonal_order[2]
S = fitted_model.seasonal_order[3]

# p, d, q, P, D, Q, S values are obtained from auto_arima without setting 'm=365'!
p = 1
d = 0
q = 0
P = 0
D = 0
Q = 0
S = 365

# # p, d, q, P, D, Q, S values are borrowed from the monthly model below
# p = 3
# d = 1
# q = 0
# P = 1
# D = 0
# Q = 0
# S = 365

# Initialize the ARIMA model with the initial training data
model = pm.ARIMA(order=(p, d, q), seasonal_order=(P, D, Q, S))
model.fit(train_data_detr_shortened)

# Rolling-forecast loop
for i in range(n_forecast_steps):
    # Make a one-step ahead forecast
    forecast, conf_int = model.predict(n_periods=1, return_conf_int=True)
    
    # Inverse-transform the forecast and actual value by adding the cumulative sum
    last_observed_value = train_data_shortened.iloc[-1]
    forecast_inverse = last_observed_value + forecast[0]
    actual_value_inverse = last_observed_value + test_data.iloc[i]
    
    # Store the inverse-transformed forecast and actual value in the respective lists
    forecasts.append(forecast_inverse)
    actual_values.append(actual_value_inverse)
    
    # Update the model with the true value (actual value) for the current step
    model.update(test_data.iloc[i:i+1])

# Create a DataFrame to store the forecasts and actual values
rolling_forecast_df = pd.DataFrame({
    'Forecast': forecasts,
    'Actual': actual_values
}, index=test_data_detr.index)

fcv_series = rolling_forecast_df['Forecast']

# Plot the predicted solar and original solar generation
plt.figure(figsize=(12,5), dpi=100)
plt.plot(train_data_shortened[1380:], label='Training data', color='black')
plt.plot(test_data, label='Actual solar energy generation', color='blue')
plt.plot(fcv_series, label='Predicted solar energy generation', color='red')
plt.title('Prediction for daily solar (with de-trending)')
plt.xlabel('Time')
plt.ylabel('Daily solar energy generation [GWh]')
# plt.xticks([])
plt.legend(loc='upper left', fontsize=8)
#plt.show()

# Evaluate the model by calculating RMSE
rms_auto_arima = sqrt(mean_squared_error(test_data.values, fcv_series))
print("Auto-Arima RMSE :- " + str(round(rms_auto_arima,3)))

In [ ]:
print(rolling_forecast_df.shape)
print(test_data_detr.shape)
info_describe(rolling_forecast_df, 5)
rolling_forecast_df.head(5)

#### 'df_monthly'

In [ ]:
df_daily.head(2)

In [ ]:
# Create a new DataFrame for monthly energy generation

df_temp = df_daily.copy()

monthly_columns = ['dt_date_hour'] + energy_types + ['solar_gwh'] + ['total_energy_generation_gwh']

df_monthly = df_temp[monthly_columns].groupby(pd.Grouper(key='dt_date_hour', freq='M')).sum().reset_index()

In [ ]:
df_monthly.head(2)

In [ ]:
# Create new features about datetime based on column 'dt_date_hour'

df_monthly['dt_year'] = df_monthly['dt_date_hour'].dt.year
df_monthly['dt_month'] = df_monthly['dt_date_hour'].dt.month
# df_monthly['dt_week'] = df_monthly['dt_date_hour'].dt.isocalendar().week
# df_monthly['dt_week'] = df_monthly['dt_week'].astype('int32')
# df_monthly['dt_dayofweek'] = df_monthly['dt_date_hour'].dt.dayofweek
# df_monthly['dt_dayofmonth'] = df_monthly['dt_date_hour'].dt.day
# df_monthly['dt_dayofyear'] = df_monthly['dt_date_hour'].dt.dayofyear
# df_monthly['dt_hour'] = df_monthly['dt_date_hour'].dt.hour
df_monthly['dt_monthname'] = df_monthly['dt_date_hour'].dt.month_name()
# df_monthly['dt_dayname'] = df_monthly['dt_date_hour'].dt.day_name()
# df_monthly['dt_date'] = df_monthly['dt_date_hour'].dt.date
df_monthly['dt_date'] = df_monthly['dt_date_hour'].dt.strftime('%Y-%m')

In [ ]:
# Delete the row with the entry '2023-07' in the 'dt_date' column

entry_to_delete = '2023-07'
df_monthly = df_monthly[df_monthly['dt_date'] != entry_to_delete]

# Reset the index
df_monthly = df_monthly.reset_index(drop=True)

In [ ]:
df_monthly.tail(2)

In [ ]:
# Monthly solar energy production in Germany

plt.figure(figsize=(6,4), dpi=100)

fig, ax = plt.subplots()

# Create the line plot using Seaborn
sns.lineplot(data=df_monthly, x='dt_date', y='solar_gwh')

# Get the current axis
# ax = plt.gca()

# Set the x-axis ticks for every 6 increments of x-values
ax.set_xticks(df_monthly['dt_date'][::6])

# Rotate the x-axis tick labels for better readability
plt.xticks(rotation=45)

ax.set(title="Germany's monthly solar energy generation", xlabel='Timestamp', ylabel='Solar energy generation [GWh]')

plt.show()

In [ ]:
df_monthly['solar_twh'] = df_monthly['solar_gwh'] / 1000.0

#### 'df_monthly_idx'

In [ ]:
# Set 'dt_date' as the index of the DataFrame 'df_monthly_idx'
df_monthly_idx = df_monthly.set_index('dt_date', inplace=False)
df_monthly_idx.head()

In [ ]:
print(df_monthly_idx.index.min(), df_monthly_idx.index.max())
print(len(df_monthly_idx))

In [ ]:
# Decompose 'df_monthly_idx' using 'seasonal_decompose'

# Use the parameter 'period=12' to extract the yearly seasonality

decomposition = seasonal_decompose(x=df_monthly_idx['solar_twh'], 
                                   model='multiplicative',
                                   two_sided=True,
                                   period=12)

fig, ax = plt.subplots(4, 1, figsize=(12, 12), constrained_layout=True, sharex='all')

decomposition.observed.plot(c=sns_c[0], ax=ax[0])
ax[0].set(title="Germany's monthly solar energy generation [TWh]",xlabel='Date')

decomposition.trend.plot(c=sns_c[1], ax=ax[1])
ax[1].set(title='Trend component',xlabel='Date')

decomposition.seasonal.plot(c=sns_c[2], ax=ax[2])
ax[2].set(title='Seasonal component',xlabel='Date')

decomposition.resid.plot(c=sns_c[3], ax=ax[3])
ax[3].set(title='Residual component',xlabel='Date')


# df_monthly_idx.index = pd.to_datetime(df_monthly_idx.index)

# # Set the desired number of major x-ticks
# desired_major_ticks = 6

# # Calculate the appropriate interval between major ticks
# interval = len(df_monthly_idx) // (desired_major_ticks - 1)

# # Customize the x-axis tick resolution to display major ticks at the calculated interval
# ax[0].xaxis.set_major_locator(mdates.IndexLocator(base=interval, offset=0))

# # Format the date ticks on the x-axis to show only the year and month ('YYYY-MM')
# date_format = mdates.DateFormatter('%Y-%m')
# ax[0].xaxis.set_major_formatter(date_format)

fig.set_size_inches(10, 5)

In [ ]:
# 2nd way of plotting decomposed 'df_monthly_idx' using 'seasonal_decompose'

# Perform the seasonal decomposition
decomposition = seasonal_decompose(x=df_monthly_idx['solar_twh'],
                                   model='multiplicative',
                                   two_sided=True,
                                   period=12)

# Create the subplots with a shared x-axis
# fig, ax = plt.subplots(4, 1, figsize=(12, 12), constrained_layout=True, sharex='all')

fig, ax = plt.subplots(figsize=(12, 6))

# Plot the 'observed' trace on the first subplot
ax.plot_date(df_monthly_idx.index, decomposition.observed, c=sns_c[0], linestyle='-')
ax.set(title="Germany's monthly solar energy generation [TWh]", xlabel='Date')

# ax[0].plot_date(df_monthly_idx.index, decomposition.observed, c=sns_c[0], linestyle='-')
# ax[0].set(title="Germany's monthly solar energy generation [TWh]", xlabel='Date')

# ax[1].plot_date(df_monthly_idx.index, decomposition.trend, c=sns_c[1], linestyle='-')
# ax[1].set(title='Trend', xlabel='Date')

# ax[2].plot_date(df_monthly_idx.index, decomposition.seasonal, c=sns_c[2], linestyle='-')
# ax[2].set(title='Seasonal', xlabel='Date')

# ax[3].plot_date(df_monthly_idx.index, decomposition.resid, c=sns_c[3], linestyle='-')
# ax[3].set(title='Residual', xlabel='Date')


# Manually set the x-axis limits to the range of the index of the DataFrame
ax.set_xlim(df_monthly_idx.index.min(), df_monthly_idx.index.max())

# Show the actual dates on the x-axis
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

# Set the number of minor ticks to have more frequent ticks
ax.xaxis.set_minor_locator(mdates.WeekdayLocator(byweekday=(mdates.MO)))

# Show the plot
plt.show()


In [ ]:
# 3rd way of plotting decomposed 'df_monthly_idx' using 'seasonal_decompose'

# Perform the seasonal decomposition
decomposition = seasonal_decompose(x=df_monthly_idx['solar_twh'],
                                   model='multiplicative',
                                   two_sided=True,
                                   period=12)

# Create the subplots with a shared x-axis
fig, ax = plt.subplots(4, 1, figsize=(12, 12), constrained_layout=True, sharex='all')

# Plot the 'observed' trace on the first subplot as a line
ax[0].plot(df_monthly_idx.index, decomposition.observed, c=sns_c[0], linestyle='-')
ax[0].set(title='Signal', xlabel='Date')

ax[1].plot(df_monthly_idx.index, decomposition.trend, c=sns_c[1], linestyle='-')
ax[1].set(title='Trend', xlabel='Date')

ax[2].plot(df_monthly_idx.index, decomposition.seasonal, c=sns_c[2], linestyle='-')
ax[2].set(title='Seasonal', xlabel='Date')

ax[3].plot(df_monthly_idx.index, decomposition.resid, c=sns_c[3], linestyle='-')
ax[3].set(title='Residual', xlabel='Date')


# Manually set the x-axis limits to the range of the index of the DataFrame
ax[0].set_xlim(df_monthly_idx.index.min(), df_monthly_idx.index.max())

# Show the actual dates on the x-axis
ax[0].xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

# Set the number of minor ticks to have more frequent ticks (corresponding to 1 month increment)
ax[0].xaxis.set_minor_locator(mdates.MonthLocator(interval=1))


# # Manually set the x-axis limits to the range of the index of the DataFrame
# ax[0].set_xlim(df_monthly_idx.index.min(), df_monthly_idx.index.max())

# # Reduce the number of major x-ticks for better visibility
# ax[0].xaxis.set_major_locator(mdates.YearLocator())

# # Show the actual dates on the x-axis
# ax[0].xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

# # Set the number of minor ticks to have more frequent ticks (corresponding to 1 month increment)
# ax[0].xaxis.set_minor_locator(mdates.MonthLocator(interval=3))

# # Explicitly set the 'minor' property to show the minor x-ticks
# ax[0].xaxis.set_minor_formatter(mdates.DateFormatter('%m'))


plt.show()


In [ ]:
# De-trending 'df_monthly_idx'

df_monthly_idx['solar_twh_detrend'] = (df_monthly_idx['solar_twh'] - df_monthly_idx['solar_twh'].shift(12))
df_monthly_idx.head(2)

In [ ]:
# Test for stationarity after de-trending 'df_monthly_idx'

def stationarity_test_dt_m(energy_generation):
    
    # Calculate rolling mean and rolling standard deviation
    rolling_mean = energy_generation.rolling(12).mean()
    rolling_std_dev = energy_generation.rolling(12).std()
  
    # Plot the statistics
    plt.figure(figsize=(12,6))

    fig, ax = plt.subplots()

    plt.plot(rolling_mean, label='Rolling Mean',linewidth=2.0)
    plt.plot(rolling_std_dev, label = 'Rolling Std Dev',linewidth=2.0)
    plt.plot(energy_generation,label='De-Trended Time Series')
    # plt.xticks([])
    plt.legend(loc='best')
    plt.title('Rolling Mean and Standard Deviation')

    # Set the x-axis ticks for every 6 increments of x-values
    # ax.set_xticks(df_monthly['dt_date'][::6])

    # Rotate the x-axis tick labels for better readability
    plt.xticks(rotation=45)

    # plt.tight_layout()
    
    # ADF test
    print("ADF Test:")
    adf_test = adfuller(energy_generation,autolag='AIC')
    print('Null Hypothesis: Not Stationary')
    print('ADF Statistic: %f' % adf_test[0])
    print('p-value: %f' % adf_test[1])
    print('----'*10)
    
    # KPSS test
    print("KPSS Test:")
    kpss_test = kpss(energy_generation, regression='c', nlags='legacy', store=False)
    print('Null Hypothesis: Stationary')
    print('KPSS Statistic: %f' % kpss_test[0])
    print('p-value: %f' % kpss_test[1])
    print('----'*10)
    
stationarity_test_dt_m(df_monthly_idx['solar_twh_detrend'].dropna())

# Partial Autocorrelation Plot
pacf = plot_pacf(df_monthly_idx['solar_twh_detrend'].dropna(), lags=12)

#### Data splitting for 'df_monthly_idx'

In [ ]:
# Dropna for 'solar_twh_detrend' of df_monthly_idx

print(df_monthly_idx['solar_twh_detrend'].head(2))
print(df_monthly_idx['solar_twh_detrend'].tail(2))

# s2 = df_monthly_idx['solar_twh_detrend'].dropna()

s2_detr = df_monthly_idx['solar_twh_detrend']
s2_orig = df_monthly_idx['solar_twh']

print(s2_detr.shape, s2_orig.shape)

print(s2_detr.tail(7))

In [ ]:
print(s2_orig.index[101])
print(s2_detr.index[0])

In [ ]:
# Split data into train and test set for 'df_monthly_idx'

total_rows = s2_orig.shape
test_rows = df_monthly_idx.query('dt_year==2023 or dt_year==2022 or dt_year==2021')['dt_year'].count()
split_ratio = (total_rows - test_rows) * 1.0 / total_rows
print(f"Split ratio: {split_ratio}")

df_arima_m = s2_orig
df_arima_detr_m = s2_detr
train_test_split_point = int(len(df_arima_m)*split_ratio)
train_data_m, test_data_m = df_arima_m[:train_test_split_point], df_arima_m[train_test_split_point:]
train_data_detr_m, test_data_detr_m = df_arima_detr_m[:train_test_split_point].dropna(), df_arima_detr_m[train_test_split_point:]

print(f"Original train data: {train_data_m.index[0]} - {train_data_m.index[train_test_split_point-1]}")
print(f"Original test data: {test_data_m.index[0]} - {test_data_m.index[-1]}")

print(f"De-trended train data: {train_data_detr_m.index[0]} - {train_data_detr_m.index[-1]}")
print(f"De-trended test data: {test_data_detr_m.index[0]} - {test_data_detr_m.index[-1]}")

# Plotting the train and test set

plt.figure(figsize=(10,6))
plt.xlabel('Year (after de-trending)')
plt.ylabel('Monthly solar energy generation (after de-trending)')
# plt.xticks([])
plt.plot(train_data_detr_m, 'red', label='Train data')
plt.plot(test_data_detr_m, 'black', label='Test data')

# Rotate the x-axis tick labels for better readability
plt.xticks(rotation=45)
plt.grid('both')

plt.legend()

#### A working ARIMA model for 'df_monthly_idx'

In [ ]:
# Trial-and-error: ARIMA with rolling-forecast for 'df_arima_detr_m' (i.e., monthly solar generation)

# It works!!!

import pmdarima as pm

arima_model = auto_arima(train_data_detr_m,
                      start_p=0, start_q=0,
                      max_p=6, max_q=6,
                      test='adf',        
                      trace=True,
                      alpha=0.05,
                      scoring='mse',
                      suppress_warnings=True,
                      seasonal=True,
                      m=12,  
                      n_jobs=-1,
                      stepwise=True
                      )

# sarima_model = pm.auto_arima(train_data, seasonal=True, m=365, max_p=max_p, max_d=max_d, max_q=max_q,
#                              max_P=max_P, max_D=max_D, max_Q=max_Q, max_S=max_S, start_S=min_S,
#                              stepwise=True, trace=True, 
#                              test='adf', scoring='mse', suppress_warnings=True)

# Fit the final model with the order
fitted_model = arima_model.fit(train_data_detr_m) 
print(fitted_model.summary())

# Define the rolling window size (e.g., 365 days)
window_size = 12
n_forecast_steps = len(test_data_detr_m)

# Initialize lists to store forecasts and actual values
forecasts = []
actual_values = []

# Obtain ARIMA model orders from the fitted model
p = fitted_model.order[0]
d = fitted_model.order[1]
q = fitted_model.order[2]
P = fitted_model.seasonal_order[0]
D = fitted_model.seasonal_order[1]
Q = fitted_model.seasonal_order[2]
S = fitted_model.seasonal_order[3]

# Initialize the ARIMA model with the initial training data
model = pm.ARIMA(order=(p, d, q), seasonal_order=(P, D, Q, S))
model.fit(train_data_detr_m)

# Rolling-forecast loop
for i in range(n_forecast_steps):
    # Make a one-step ahead forecast
    forecast, conf_int = model.predict(n_periods=1, return_conf_int=True)
    
    # Inverse-transform the forecast and actual value by adding the cumulative sum
    last_observed_value = train_data_m.iloc[-1]
    forecast_inverse = last_observed_value + forecast[0]
    actual_value_inverse = last_observed_value + test_data_m.iloc[i]
    
    # Store the inverse-transformed forecast and actual value in the respective lists
    forecasts.append(forecast_inverse)
    actual_values.append(actual_value_inverse)
    
    # Update the model with the true value (actual value) for the current step
    model.update(test_data_m.iloc[i:i+1])

# Create a DataFrame to store the forecasts and actual values
rolling_forecast_df = pd.DataFrame({
    'Forecast': forecasts,
    'Actual': actual_values
}, index=test_data_detr_m.index)

fcv_series = rolling_forecast_df['Forecast']


# Plot the forecasted and the actual monthly solar generation

train_data_m.index = pd.to_datetime(train_data_m.index)
test_data_m.index = pd.to_datetime(test_data_m.index)
fcv_series.index = pd.to_datetime(fcv_series.index)

plt.figure(figsize=(12,5), dpi=100)
plt.plot(train_data_m[:], label='Training', color='black')
plt.plot(test_data_m, label='Actual solar energy generation', color='blue')
plt.plot(fcv_series,label='Predicted solar energy generation', color='red')

# Set the x-axis major locator to display ticks every 6 months
locator = MonthLocator(interval=6)
plt.gca().xaxis.set_major_locator(locator)

# Get the first and last dates of the data
first_date = pd.to_datetime(train_data_m.index[0])
last_date = pd.to_datetime(test_data_m.index[-1])

# Set the x-axis limits to [2015-01, 2023-06]
plt.gca().set_xlim(left=first_date.replace(month=1, day=1), right=last_date.replace(month=6, day=30))

# Rotate the x-axis tick labels for better readability
plt.xticks(rotation=45)

plt.title('Prediction for monthly solar generation')
plt.xlabel('Time')
plt.ylabel('Monthly solar energy generation [TWh]')
# plt.xticks([])
plt.legend(loc='upper left')
plt.show()

# Evaluate the model by calculating RMSE
rms_auto_arima = sqrt(mean_squared_error(test_data_m.values, fcv_series))
print("Auto-Arima RMSE :- " + str(round(rms_auto_arima,3)))

#### Seasonal ARIMA: more organized procedure

In [ ]:
# Seasonal ARIMA: hyperparameter tuning with fixed 'alpha' (e.g., 0.001)

# Define the maximum values for the hyperparameter search space
max_p = 6  # maximum value for p (non-seasonal AR order)
max_d = 2  # maximum value for d (non-seasonal differencing order)
max_q = 6  # maximum value for q (non-seasonal MA order)
max_P = 2  # maximum value for P (seasonal AR order)
max_D = 2  # maximum value for D (seasonal differencing order)
max_Q = 2  # maximum value for Q (seasonal MA order)
max_S = 12  # maximum value for S (seasonal periodicity)

# Find the optimal SARIMA hyperparameters using auto_arima with specified hyperparameter ranges
sarima_model = pm.auto_arima(train_data_detr_m, seasonal=True, m=12, max_p=max_p, max_d=max_d, max_q=max_q,
                             max_P=max_P, max_D=max_D, max_Q=max_Q, max_S=max_S,
                             stepwise=True, trace=True, test='adf', scoring='mse', suppress_warnings=True,
                             alpha=0.05, n_jobs=-1)

# Get the best SARIMA hyperparameters
p, d, q = sarima_model.order
P, D, Q, S = sarima_model.seasonal_order

# Initialize the SARIMA model with the best hyperparameters
model = pm.ARIMA(order=(p, d, q), seasonal_order=(P, D, Q, S))

# Define the rolling window size (e.g., 12 months)
window_size = 12
n_forecast_steps = len(test_data_detr_m)

# Initialize lists to store forecasts and actual values
forecasts = []
actual_values = []

# Initialize the SARIMA model with the initial training data
model.fit(train_data_detr_m)

print(model.summary())

# Rolling-forecast loop
for i in range(n_forecast_steps):
    # Make a one-step ahead forecast
    forecast, conf_int = model.predict(n_periods=1, return_conf_int=True)
    
    # Inverse-transform the forecast and actual value by adding the cumulative sum
    last_observed_value = train_data_m.iloc[-1]
    forecast_inverse = last_observed_value + forecast[0]
    actual_value_inverse = last_observed_value + test_data_m.iloc[i]
    
    # Store the inverse-transformed forecast and actual value in the respective lists
    forecasts.append(forecast_inverse)
    actual_values.append(actual_value_inverse)
    
    # Update the model with the true value (actual value) for the current step
    model.update(test_data_m.iloc[i:i+1])

# Create a DataFrame to store the forecasts and actual values
rolling_forecast_df = pd.DataFrame({
    'Forecast': forecasts,
    'Actual': actual_values
}, index=test_data_detr_m.index)

fcv_series = rolling_forecast_df['Forecast']

# Evaluate the model by calculating RMSE
rms_sarima = sqrt(mean_squared_error(test_data_m.values, fcv_series))
print('\n')
print("Sarima RMSE :- " + str(round(rms_sarima,3)))
print('\n')

# Print test values and rolling forecasts
for i in range(n_forecast_steps):
    print(f"Timestamp: {test_data_m.index[i].strftime('%Y-%m')}, Test: {test_data_m[i]: 4.3f}, Forecast: {rolling_forecast_df['Forecast'][i]: 4.3f}")


In [ ]:
# Tune 'alpha' value as well

# alpha: 0.005, RMSE: -1.816
# alpha: 0.01,  RMSE: -1.603
# alpha: 0.05,  RMSE: -1.603
# alpha: 0.0833,RMSE: -1.603
# alpha: 0.1,   RMSE: -1.603
# alpha: 0.2,   RMSE: -2.772

# Note: AIC is the default scoring metric!

# Define the maximum values for the hyperparameter search space
max_p = 6  # maximum value for p (non-seasonal AR order)
max_d = 2  # maximum value for d (non-seasonal differencing order)
max_q = 6  # maximum value for q (non-seasonal MA order)
max_P = 2  # maximum value for P (seasonal AR order)
max_D = 2  # maximum value for D (seasonal differencing order)
max_Q = 2  # maximum value for Q (seasonal MA order)
max_S = 12  # maximum value for S (seasonal periodicity)

# Define the search space for the alpha parameter
alpha_values = [0.0005, 0.001, 0.002, 0.005, 0.01, 0.05, 0.1]

# Create the sarimax_kwargs dictionary with the alpha parameter
sarimax_kwargs = {'alpha': alpha_values}

# Find the optimal SARIMA hyperparameters using auto_arima with specified hyperparameter ranges
# sarima_model = pm.auto_arima(train_data_detr_m, seasonal=True, m=12, max_p=max_p, max_d=max_d, max_q=max_q,
#                              max_P=max_P, max_D=max_D, max_Q=max_Q, max_S=max_S,
#                              stepwise=True, trace=True, test='adf', scoring='mse', suppress_warnings=True,
#                              sarimax_kwargs=sarimax_kwargs, n_jobs=-1, information_criterion='aic')

sarima_model = pm.auto_arima(train_data_detr_m, seasonal=True, m=12, max_p=max_p, max_d=max_d, max_q=max_q,
                             max_P=max_P, max_D=max_D, max_Q=max_Q, max_S=max_S,
                             stepwise=True, trace=True, test='adf', scoring='mse', suppress_warnings=True,
                             alpha=0.05, n_jobs=-1, information_criterion='aic')

# Get the best SARIMA hyperparameters
p, d, q = sarima_model.order
P, D, Q, S = sarima_model.seasonal_order

# Initialize the SARIMA model with the best hyperparameters
model = pm.ARIMA(order=(p, d, q), seasonal_order=(P, D, Q, S))

# Define the rolling window size (e.g., 12 months)
window_size = 12
n_forecast_steps = len(test_data_detr_m)

# Initialize lists to store forecasts and actual values
forecasts = []
actual_values = []

# Initialize the SARIMA model with the initial training data
model.fit(train_data_detr_m)

print(model.summary())

# Extract the seasonal parameters from the SARIMA model
seasonal_params = sarima_model.get_params()['seasonal_order']

# Access the value of the alpha parameter in the seasonal differencing part of the SARIMA model
m_value = seasonal_params[3]

print("\n'm' value in the seasonal differencing part of SARIMA:", m_value)

# Access the coefficients of the SARIMA model
coefficients = sarima_model.params
print("Coefficients of the SARIMA model:", coefficients)


# Rolling-forecast loop
for i in range(n_forecast_steps):
    # Make a one-step ahead forecast
    forecast, conf_int = model.predict(n_periods=1, return_conf_int=True)
    
    # Inverse-transform the forecast and actual value by adding the cumulative sum
    last_observed_value = train_data_m.iloc[-1]
    forecast_inverse = last_observed_value + forecast[0]
    actual_value_inverse = last_observed_value + test_data_m.iloc[i]
    
    # Store the inverse-transformed forecast and actual value in the respective lists
    forecasts.append(forecast_inverse)
    actual_values.append(actual_value_inverse)
    
    # Update the model with the true value (actual value) for the current step
    model.update(test_data_m.iloc[i:i+1])

# Create a DataFrame to store the forecasts and actual values
rolling_forecast_df = pd.DataFrame({
    'Forecast': forecasts,
    'Actual': actual_values
}, index=test_data_detr_m.index)

fcv_series = rolling_forecast_df['Forecast']

# Evaluate the model by calculating RMSE
rms_sarima = sqrt(mean_squared_error(test_data_m.values, fcv_series))
print('\n')
print("Sarima RMSE :- " + str(round(rms_sarima,3)))
print('\n')

# Print test values and rolling forecasts
for i in range(n_forecast_steps):
    print(f"Timestamp: {test_data_m.index[i].strftime('%Y-%m')}, Test: {test_data_m[i]: 4.3f}, Forecast: {rolling_forecast_df['Forecast'][i]: 4.3f}")

In [ ]:
# Plot forecasting result using Seaborn

train_data_m.index = pd.to_datetime(train_data_m.index)
test_data_m.index = pd.to_datetime(test_data_m.index)
fcv_series.index = pd.to_datetime(fcv_series.index)

# Set the font size for all elements on the plot (e.g., axes labels, tick labels, legend, etc.)
sns.set(font_scale=1.0)

plt.figure(figsize=(10, 5))  # Set the figure size if needed
sns.lineplot(data=train_data_m, color='black', label='Historical data')
sns.lineplot(data=test_data_m, color='black', label='Actual solar energy generation', linestyle='--')
sns.lineplot(data=fcv_series, color='red', label='Predicted solar energy generation\n(average deviation: ~1.6 TWh)')

# Set the x-axis major locator to display ticks every 6 months
locator = MonthLocator(interval=6)
plt.gca().xaxis.set_major_locator(locator)

# Get the first and last dates of the data
first_date = pd.to_datetime(train_data_m.index[0])
last_date = pd.to_datetime(test_data_m.index[-1])

# Set the x-axis limits to [2015-01, 2023-06]
plt.xlim(left=first_date.replace(month=1, day=1), right=last_date.replace(month=6, day=30))

# Rotate the x-axis tick labels for better readability
plt.xticks(rotation=45)

plt.title('Prediction for monthly solar energy generation of Germany')
plt.xlabel('Date')
plt.ylabel('Monthly solar energy generation [TWh]')
plt.legend(loc='upper left')

# Display the plot
plt.show()


# Other time series forecasting models

### Naive forecast

In [ ]:
# Naive forecast / persistence

# walk-forward validation
history = [x for x in train_data]
predictions = list()
for i in range(len(test_data)):
# predict
    yhat = history[-1]
    predictions.append(yhat)
    obs = test_data[i]
    history.append(obs)
    print('>Predicted=%.3f, Expected=%3.f' % (yhat, obs))
# report performance
rmse = sqrt(mean_squared_error(test_data, predictions)) 
print('RMSE: %.3f' % rmse)

### Some supportive plots for df_daily_idx['solar_gwh']

In [ ]:
df_daily_idx['solar_gwh'].describe()

In [ ]:
df_daily_idx.head(2)
df_daily_idx.tail(2)

In [ ]:
# Seasonal line plots

# Convert the 'dt_date' index to pandas datetime object
df_daily_idx.index = pd.to_datetime(df_daily_idx.index)

# Group the 'solar_gwh' column by 'dt_year'
yearly_groups = df_daily_idx['solar_gwh'].groupby(df_daily_idx['dt_year'])

# Create subplots with one column and as many rows as there are years
num_years = len(yearly_groups)
fig, axs = plt.subplots(num_years, 1, figsize=(6, 18), sharey=True)

# Loop through the yearly groups and plot the data in each subplot
for i, (year, group_data) in enumerate(yearly_groups):
    # Plot the data in the current subplot
    axs[i].plot(group_data.index.dayofyear, group_data.values)
    axs[i].set_title(f'Year {year}')
    axs[i].set_xlabel('Day of the Year')
    axs[i].set_ylabel('Solar [GWh]')
    axs[i].set_xlim(1, 366)  # Set the same xlim for each subplot (366 days)

# Adjust the layout of subplots and display the plot
plt.tight_layout()
plt.show()

In [ ]:
# Create the density plot for 'solar_gwh'

# plt.figure(figsize=(8,8), dpi=100)
# plt.figure(1)

# plt.subplot(211)
# df_daily_idx['solar_gwh'].hist()
# plt.subplot(212)
# df_daily_idx['solar_gwh'].plot(kind='kde')
# plt.show()

# Create the density plot for 'solar_gwh' using Seaborn
plt.figure(figsize=(6, 4))
sns.histplot(df_daily_idx['solar_gwh'], kde=True)
plt.title('Density Plot for Daily Solar Energy [GWh]')
plt.xlabel('Solar [GWh]')
plt.ylabel('Density')
plt.show()

In [ ]:
# Create the box plot for daily solar generation using Seaborn

plt.figure(figsize=(6, 4))
sns.boxplot(x='dt_year', y='solar_gwh', data=df_daily_idx)
plt.title('Distribution of Daily Solar Generation')
plt.xlabel('Year')
plt.ylabel('Daily Solar Generation [GWh]')
plt.show()

In [ ]:
info_describe(df_daily_idx, 2)

# Regression modeling for monthly solar generation

## Concatenate DataFrames

In [ ]:
print(df_monthly.shape)
df_monthly.head(103)

In [ ]:
df_monthly.info()

In [ ]:
info_describe(df_monthly, 2)

In [ ]:
file_path_c = '../output/df_monthly_solar_capacity.csv'
file_path_p = '../output/df_monthly_electricity_price.csv'
file_path_w = '../output/df_monthly_weather_average.csv'

df_m_c_temp = pd.read_csv(file_path_c, delimiter=',')
df_m_p_temp = pd.read_csv(file_path_p, delimiter=',')
df_m_w_temp = pd.read_csv(file_path_w, delimiter=',')

In [ ]:
# Drop a problematic column

df_m_c = df_m_c_temp.drop(['month_dot_year'], axis=1)
print(df_m_c.columns)
df_m_p = df_m_p_temp.drop(['month_dot_year'], axis=1)
print(df_m_p.columns)

# Drop a few redundant columns

df_m_w = df_m_w_temp.drop(['year', 'month', 'year_month'], axis=1)
print(df_m_p.columns)

In [ ]:
df_m_all = pd.concat([df_monthly, df_m_c, df_m_p, df_m_w], axis=1)

In [ ]:
df_m_all.info()

In [ ]:
for col in df_m_all.columns:
    print(f"{col:>60} : {df_m_all[col].dtype}")

In [ ]:
info_describe(df_m_all, 2)

In [ ]:
# Overview of missing values

msno.matrix(df_m_all, labels=True)

In [ ]:
df_m_all['solar_twh'] = df_m_all['solar_gwh'] / 1000.0
df_m_all['solar_twh'].describe().T
info_describe(df_m_all, 2)

In [ ]:
# # Profiling (run time: 1m 24s)

# # Generate profile report using ProfileReport
# df_m_all_profile = ProfileReport(df_m_all, title="Profiling Report")
# # Export profile report
# df_m_all_profile.to_file("../output/df_m_all_profile.html")

## Baseline model based on a heuristic selection of features (without scaling)

### Select features

In [ ]:
print(df_m_all.columns)

In [ ]:
input_features_m_init = ['dt_month', 'solar_gw', 'spot_market_price_eur_per_mwh',
       'co2_emission_allowances_auction_eur_per_tco2', 
       'mean_air_temperature_de_c', 'mean_sunshine_duration_de_minute',
       'mean_precipitation_de_mm']

target_feature_m = ['solar_twh']

In [ ]:
X = df_m_all[input_features_m_init]

y = df_m_all[target_feature_m]

### Split dataset

In [ ]:
# Split X and y into training and testing datasets

# RSEED = 99

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=18*1.0/102, random_state=RSEED)

# Show the results of the split
print("Training set has {} samples.".format(X_train.shape[0]))
print("Testing set has {} samples.".format(X_test.shape[0]))
print(f"y_train: {y_train.shape}")
print(f"y_test: {y_test.shape}")

### Train, predict, and evaluate (random forest, adaboost, xgboost)

In [ ]:
# Function to perform training with designated regression method, and then make predictions for the test dataset

def train_and_predict(X_train, X_test, y_train, clf_method='xgboost'):
    # Create the classifier object
    if clf_method == 'logreg':
        clf = LogisticRegression(random_state=RSEED)
    elif clf_method == 'rforest':
        clf = RandomForestRegressor(random_state=RSEED)
    elif clf_method == 'xgboost':
        clf = XGBRegressor(random_state = RSEED)
    else:
        clf = AdaBoostRegressor(random_state=RSEED)

    # Train and predict
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    # print("Predicted values:") 
    # print(y_pred) 
    return y_pred

In [ ]:
# Train random forest, AdaBoost, and XGBoost models on the train data

y_pred_rforest = train_and_predict(X_train, X_test, y_train, 'rforest')
y_pred_adaboost = train_and_predict(X_train, X_test, y_train, 'adaboost')
y_pred_xgboost = train_and_predict(X_train, X_test, y_train, 'xgboost')

In [ ]:
# Evaluate different models by comparing r2_score, RMSE, and MAE

# Function to calculate RMSE and r2_score
def cal_scores(y_test, y_pred, clf_method_name='xgboost'): 
    
    print("====="*10)
    print(f"'{clf_method_name}'")
    print("-----"*2)
    
    print(f"{'r2_score':<12}{'RMSE':<12}{'MAE':<12}")
    print(f"{r2_score(y_test,y_pred):<12.3f}{mean_squared_error(y_test,y_pred,squared=False):<12.3f}{mean_absolute_error(y_test,y_pred):<12.3f}")

In [ ]:
cal_scores(y_test, y_pred_rforest, 'rforest')
cal_scores(y_test, y_pred_adaboost, 'adaboost')
cal_scores(y_test, y_pred_xgboost, 'xgboost')

In [ ]:
# Function to perform training with designated regression method, and then make predictions for both the train and the test datasets

def train_and_predict_double(X_train, X_test, y_train, clf_method='xgboost'):
    # Create the classifier object
    if clf_method == 'logreg':
        clf = LogisticRegression(random_state=RSEED)
    elif clf_method == 'rforest':
        clf = RandomForestRegressor(random_state=RSEED)
    elif clf_method == 'xgboost':
        clf = XGBRegressor(random_state = RSEED)
    else:
        clf = AdaBoostRegressor(random_state=RSEED)

    # Train and predict
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    y_train_pred = clf.predict(X_train)

    return y_pred, y_train_pred

In [ ]:
# Train random forest, AdaBoost, and XGBoost models on the train data

y_pred_rforest, y_train_pred_rforest = train_and_predict_double(X_train, X_test, y_train, 'rforest')
y_pred_adaboost, y_train_pred_adaboost = train_and_predict_double(X_train, X_test, y_train, 'adaboost')
y_pred_xgboost, y_train_pred_xgboost = train_and_predict_double(X_train, X_test, y_train, 'xgboost')

In [ ]:
# Evaluate different models by comparing r2_score, RMSE, and MAE

# Function to calculate RMSE and r2_score
def cal_scores_double(y_test, y_pred, y_train, y_train_pred, clf_method_name='xgboost'): 
    
    print("====="*10)
    print(f"'{clf_method_name}'")
    print("-----"*2)
    
    print(f"          {'r2_score':<12}{'RMSE':<12}{'MAE':<12}")
    print(f"Train:    {r2_score(y_train,y_train_pred):<12.3f}{mean_squared_error(y_train,y_train_pred,squared=False):<12.3f}{mean_absolute_error(y_train,y_train_pred):<12.3f}")
    print(f"Test:     {r2_score(y_test,y_pred):<12.3f}{mean_squared_error(y_test,y_pred,squared=False):<12.3f}{mean_absolute_error(y_test,y_pred):<12.3f}")
    

In [ ]:
cal_scores_double(y_test, y_pred_rforest, y_train, y_train_pred_rforest, 'rforest')
cal_scores_double(y_test, y_pred_adaboost, y_train, y_train_pred_adaboost, 'adaboost')
cal_scores_double(y_test, y_pred_xgboost, y_train, y_train_pred_xgboost, 'xgboost')

### XGBoost: randomized search

In [ ]:
# Randomized search (run time: ~4m)

param_grid = {
    "subsample": list(np.arange(0.1, 0.7, 0.05)),
    "reg_lambda": [1e-5, 1e-2, 0.1, 0.5, 1, 2, 5, 10, 100],
    "reg_alpha": [1e-5, 1e-2, 0.1, 1, 5, 10, 20, 100],
    "n_estimators": list(np.arange(10, 200, 10)), # default 100
    "max_depth": list(np.arange(2, 13, 1)), # default 3
    "learning_rate": [0.0001, 0.0003, 0.001, 0.003, 0.01, 0.03, 0.05, 0.1, 0.15, 0.2, 0,3], # default 0.1
    "gamma": list(np.arange(0, 0.6, 0.05)),
    "colsample_bytree": list(np.arange(0.3, 1.0, 0.05))    
}

total_iters = 20000
cv_folds = 5

# Estimator for use in random search
estimator = XGBRegressor(random_state = RSEED)

# Create the random search model
xgb_rs = RandomizedSearchCV(estimator, param_grid, n_jobs = -1, 
                        scoring = 'r2', cv = cv_folds, 
                        n_iter = total_iters, verbose = 0, random_state=RSEED)

# Fit 
xgb_rs.fit(X_train, y_train)




# Best score:
# 0.98
# Best parameters:
# {'subsample': 0.45000000000000007, 'reg_lambda': 2, 'reg_alpha': 0.1, 'n_estimators': 140, 'max_depth': 7, 'learning_rate': 0.2, 'gamma': 0.0, 'colsample_bytree': 0.5999999999999999}

# Best score:
# 0.98
# Best parameters:
# {'subsample': 0.20000000000000004, 'reg_lambda': 0.1, 'reg_alpha': 20, 'n_estimators': 190, 'max_depth': 7, 'learning_rate': 0.1, 'gamma': 0.15000000000000002, 'colsample_bytree': 0.6499999999999999}



In [ ]:
print('Best score:\n{:.2f}'.format(xgb_rs.best_score_))
print("Best parameters:\n{}".format(xgb_rs.best_params_))

In [ ]:
# Apply best model for RS XGBoost

best_model_xgbrs = xgb_rs.best_estimator_

y_pred_xgbrs_best = best_model_xgbrs.predict(X_test)
y_train_pred_xgbrs_best = best_model_xgbrs.predict(X_train)

cal_scores_double(y_test, y_pred_xgbrs_best, y_train, y_train_pred_xgbrs_best, 'Randomized Search XGBoost: best scores')

In [ ]:
# Plotting test and predicted values

# Draw a scatter plot with regression line
plt.figure(figsize=(6, 4))
plt.scatter(y_test, y_pred_xgbrs_best, alpha=0.7)
plt.plot(np.linspace(np.min(y_test), np.max(y_test), 100), np.linspace(np.min(y_test), np.max(y_test), 100), color='red', linestyle='--')
plt.xlabel("Actual Target Values")
plt.ylabel("Predicted Target Values")
plt.title("Predicted values vs. y_test dataset")
plt.grid(True)
plt.show()

In [ ]:
# Plotting train and predicted values

# Draw a scatter plot with regression line
plt.figure(figsize=(6, 4))
plt.scatter(y_train, y_train_pred_xgbrs_best, alpha=0.7)
plt.plot(np.linspace(np.min(y_test), np.max(y_test), 100), np.linspace(np.min(y_test), np.max(y_test), 100), color='red', linestyle='--')
plt.xlabel("Actual Target Values")
plt.ylabel("Predicted Target Values")
plt.title("Predicted values vs. y_train dataset")
plt.grid(True)
plt.show()

### XGBoost: grid search

In [ ]:
# Grid search (run time: ~5m)

param_grid = {
    "subsample": list(np.arange(0.4, 0.55, 0.05)),
    "reg_lambda": [1, 2, 3],
    "reg_alpha": [0.05, 0.1, 0.2],
    "n_estimators": list(np.arange(100, 200, 20)), # default 100
    "max_depth": list(np.arange(4, 8, 1)), # default 3
    "learning_rate": [0.1, 0.2, 0.3], # default 0.1
    "gamma": list(np.arange(0, 0.1, 0.05)),
    "colsample_bytree": list(np.arange(0.5, 0.8, 0.1))    
}

cv_folds = 5

# Estimator for use in random search
estimator = XGBRegressor(random_state = RSEED)

# Create the random search model
xgb_gs = GridSearchCV(estimator, param_grid, n_jobs = -1, 
                        scoring = 'r2', cv = cv_folds, 
                        verbose = 1)

# Fit 
xgb_gs.fit(X_train, y_train)

# # Start the grid search and track progress using the custom progress bar
# with progressbar.ProgressBar(max_value=len(param_grid)) as pbar:
#     xgb_gs.fit(X_train, y_train)
#     pbar.update(1)


# Best score:
# 0.9795
# Best parameters:
# {'colsample_bytree': 0.6, 'gamma': 0.05, 'learning_rate': 0.2, 'max_depth': 5, 'n_estimators': 180, 'reg_alpha': 0.1, 'reg_lambda': 3, 'subsample': 0.4}

In [ ]:
print('Best score:\n{:.4f}'.format(xgb_gs.best_score_))
print("Best parameters:\n{}".format(xgb_gs.best_params_))

In [ ]:
# Apply best model for GS XGBoost

best_model_xgbgs = xgb_gs.best_estimator_

y_pred_xgbgs_best = best_model_xgbgs.predict(X_test)
y_train_pred_xgbgs_best = best_model_xgbgs.predict(X_train)


cal_scores_double(y_test, y_pred_xgbgs_best, y_train, y_train_pred_xgbgs_best, 'Grid Search XGBoost: best scores')

In [ ]:
# Plotting test and predicted values

# Draw a scatter plot with regression line
plt.figure(figsize=(6, 4))
plt.scatter(y_test, y_pred_xgbgs_best, alpha=0.7)
plt.plot(np.linspace(np.min(y_test), np.max(y_test), 100), np.linspace(np.min(y_test), np.max(y_test), 100), color='red', linestyle='--')
plt.xlabel("Actual Target Values")
plt.ylabel("Predicted Target Values")
plt.title("Predicted values vs. y_test dataset")
plt.grid(True)
plt.show()

In [ ]:
# Plotting train and predicted values

# Draw a scatter plot with regression line
plt.figure(figsize=(6, 4))
plt.scatter(y_train, y_train_pred_xgbgs_best, alpha=0.7)
plt.plot(np.linspace(np.min(y_test), np.max(y_test), 100), np.linspace(np.min(y_test), np.max(y_test), 100), color='red', linestyle='--')
plt.xlabel("Actual Target Values")
plt.ylabel("Predicted Target Values")
plt.title("Predicted values vs. y_train dataset")
plt.grid(True)
plt.show()

### XGBoost: feature importance (permutation-based)

In [ ]:
# Plot permutation-based feature importance

# run time: 

# Construct a DataFrame by applying 'permutation_importance' to the best model of GS XGBoost

result = permutation_importance(best_model_xgbgs, X_test, y_test, n_repeats=10, random_state=RSEED)

df_pi_xgbgs = pd.DataFrame()
df_pi_xgbgs['importances'] = result.importances_mean
df_pi_xgbgs['std'] = result.importances_std
df_pi_xgbgs['features'] = X_train.columns
df_pi_xgbgs.sort_values('importances', ascending=False)

In [ ]:
# Display the permutation-based feature importance DataFrame and plot the top-7 most important features 

# Set the (local) desired precision for floating-point numbers
# pd.options.display.float_format = '{:.5f}'.format

# Display the DataFrame
# print(df_pi_xgbgs.sort_values('importances', ascending=False))

top = 7 # Top-7

labels_df_pi_xgbgs = df_pi_xgbgs.sort_values('importances', ascending=False).head(top)['features'].tolist()
xerr_df_pi_xgbgs = df_pi_xgbgs.sort_values('importances', ascending=False).head(top)['std'].tolist()
df_pi_xgbgs.sort_values('importances', ascending=False).head(top)['importances'].plot(kind='barh',legend=False,xerr=xerr_df_pi_xgbgs)

plt.gca().set_yticklabels(labels_df_pi_xgbgs)
plt.gca().invert_yaxis()
plt.title('Feature importance of the XGBoost model')
plt.show()

### Ridge (CV) regression

In [ ]:
# Create the Ridge Regression model
ridge_model = Ridge(alpha=1.0)  # Adjust the alpha value as needed

# Fit the model to the training data
ridge_model.fit(X_train, y_train)

# Predict the target values on the test data
y_pred = ridge_model.predict(X_test)

# Calculate MSE, RMSE, and R2 on the test data
mse = mean_squared_error(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred, squared=False)
r2 = r2_score(y_test, y_pred)

print("MSE:", mse)
print("RMSE:", rmse)
print("R2 Score:", r2)

# Get the coefficients (weights) of the features
coefficients = ridge_model.coef_

# Extract the coefficients from the first row and flatten the array to a 1D array
coefficients_flat = coefficients[0]

# Create a pandas Series with feature names as the index and coefficients as values
coefficients_series = pd.Series(coefficients_flat, index=X_train.columns)

# Print the coefficients with feature names
print("Ridge Regression Coefficients:")
for feature, coef in coefficients_series.items():
    print(f"{feature}: {coef:.8f}")

In [ ]:
# Create the RidgeCV model and specify a list of candidate alpha values
alphas = [0.1, 0.2, 0.5, 1.0, 2, 5, 10, 20, 30, 50, 100, 150, 200, 500, 1000, 2000]
alphas = [0.1, 0.2, 0.5, 1.0]
alphas = [0.001, 0.01, 0.1, 1]

# Create a list to store R2 scores for each alpha
r2_scores = []

# Create the Ridge model and perform cross-validation for each alpha value
for alpha in alphas:
    ridge_model = Ridge(alpha=alpha)
    # Perform cross-validation and calculate the R2 score for each fold
    cv_r2_scores = cross_val_score(ridge_model, X_train, y_train, cv=5, scoring='r2')
    # Calculate the mean R2 score across all folds for this alpha
    mean_r2_score = np.mean(cv_r2_scores)
    # Store the mean R2 score for this alpha in the list
    r2_scores.append(mean_r2_score)
    print(f"Alpha: {alpha}, Mean R2 Score: {mean_r2_score:.4f}")

# Find the best alpha value based on the highest R2 score
best_alpha = alphas[np.argmax(r2_scores)]

# Print the best alpha and its corresponding R2 score
print("Best Alpha:", best_alpha)
print("Best R2 Score:", max(r2_scores))

# Train the final Ridge model using the best alpha
ridge_model = Ridge(alpha=best_alpha)

# Fit the model to the training data
ridge_model.fit(X_train, y_train)

# Make predictions on the test data using the tuned Ridge model
y_pred = ridge_model.predict(X_test)

# Calculate MSE, RMSE, and R2 on the test data
mse = mean_squared_error(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred, squared=False)
r2 = r2_score(y_test, y_pred)

print("MSE:", mse)
print("RMSE:", rmse)
print("R2 Score:", r2)

# Get the coefficients and intercept of the model
coefficients = ridge_model.coef_
intercept = ridge_model.intercept_

# Convert the coefficients to a nested Python list using tolist()
coefficients_list = coefficients.tolist()

# Use list comprehension to extract the floating-point values from the nested list
coefficients_float_list = [coef for coef in coefficients_list[0]]

print("Ridge Regression Coefficients:")
for feature, coef in zip(X_train.columns, coefficients_float_list):
    print(f"{feature}: {coef:.6f}")

# Display the exact fitting function
print("Exact Fitting Function:")
print(f"y = {intercept[0]:.6f}", end="")
for i, coef in enumerate(coefficients_float_list):
    print(f" + {coef:.6f} * x{i+1}", end="")
print()

### Feature scaling

In [ ]:
cols = input_features_m_init + target_feature_m

In [ ]:
df_m_all[cols].info()

In [ ]:
df_m_all[cols].describe().T

In [ ]:
info_describe(df_m_all[cols], 2)

In [ ]:
# Examine distribution of features

# Set the number of columns for the subplot grid
num_cols = 4

# Calculate the number of rows required based on the number of features and the number of columns
num_features = len(cols)
num_rows = (num_features + num_cols - 1) // num_cols

# Set the figure size to control the size of individual subplots
fig, axes = plt.subplots(num_rows, num_cols, figsize=(12, 3 * num_rows))

# Flatten the axes array in case there is only one row of subplots
axes = axes.flatten()

# Loop over the selected features to plot histograms with KDEs
for i, feature in enumerate(cols):
    ax = axes[i]

    # Plot histogram with KDE using Seaborn
    sns.histplot(df_m_all[feature], kde=True, ax=ax)

    # Add a title to each subplot
    ax.set_title(f"'{feature}'", fontsize=11)

# Hide any empty subplots if the number of features is not a multiple of num_cols
for i in range(num_features, num_rows * num_cols):
    axes[i].axis('off')

# Adjust the layout to avoid overlapping titles
plt.tight_layout()

# Show the plot
plt.show()


In [ ]:
# Scale the input features using StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

### Ridge (CV) regression with scaling

In [ ]:
# Create the RidgeCV model and specify a list of candidate alpha values

alphas = list(np.arange(0.1, 1.1, 0.1)) + [2, 5, 10, 20, 30, 50, 100, 150, 200, 500, 1000, 2000]

# Create a list to store R2 scores for each alpha
r2_scores = []

# Create the Ridge model and perform cross-validation for each alpha value
for alpha in alphas:
    ridge_model = Ridge(alpha=alpha)
    # Perform cross-validation and calculate the R2 score for each fold
    cv_r2_scores = cross_val_score(ridge_model, X_train_scaled, y_train, cv=5, scoring='r2')
    # Calculate the mean R2 score across all folds for this alpha
    mean_r2_score = np.mean(cv_r2_scores)
    # Store the mean R2 score for this alpha in the list
    r2_scores.append(mean_r2_score)
    print(f"Alpha: {alpha}, Mean R2 Score: {mean_r2_score:.4f}")

# Find the best alpha value based on the highest R2 score
best_alpha = alphas[np.argmax(r2_scores)]

# Print the best alpha and its corresponding R2 score
print("Best Alpha:", best_alpha)
print("Best R2 Score:", max(r2_scores))

# Train the final Ridge model using the best alpha
ridge_model = Ridge(alpha=best_alpha)

# Fit the model to the training data
ridge_model.fit(X_train_scaled, y_train)

# Make predictions on the test data using the tuned Ridge model
y_pred = ridge_model.predict(X_test_scaled)

# Calculate MSE, RMSE, and R2 on the test data
mse = mean_squared_error(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred, squared=False)
r2 = r2_score(y_test, y_pred)

print("MSE:", mse)
print("RMSE:", rmse)
print("R2 Score:", r2)

# Get the scaled coefficients and intercept of the model
coefficients = ridge_model.coef_
intercept = ridge_model.intercept_

# Convert the coefficients to a nested Python list using tolist()
coefficients_list = coefficients.tolist()

# Use list comprehension to extract the floating-point values from the nested list
coefficients_float_list = [coef for coef in coefficients_list[0]]

print("Ridge Regression Coefficients:")
for feature, coef in zip(X_train.columns, coefficients_float_list):
    print(f"{feature}: {coef:.6f}")

# Display the exact fitting function with the scaled coefficients and intercept
print("Exact Fitting Function with scaled coefficients and intercept:")
print(f"y = {intercept[0]:.6f}", end="")
for i, coef in enumerate(coefficients_float_list):
    print(f" + {coef:.6f} * x{i+1}", end="")
print()

## Forecasting based on (baseline) regression models

### Split dataset

In [ ]:
df_m_all[cols].head()
df_m_all.head(1)

In [ ]:
total_rows = df_m_all.shape[0]
pred_rows = df_m_all.query('dt_year==2023')['dt_year'].count()
pred_start_point = total_rows - pred_rows

print(f"Prediction dataset starts at the {pred_start_point}th row.")

In [ ]:
print(list(df_m_all.columns))
print(input_features_m_init)
print(target_feature_m)

In [ ]:
df_m_all_hr = df_m_all.copy()

In [ ]:
df_m_all_hr['mean_sunshine_duration_de_hr'] = df_m_all_hr['mean_sunshine_duration_de_minute'] / 60.0

In [ ]:
input_features_m_init = ['mean_sunshine_duration_de_hr', 'mean_air_temperature_de_c',  'mean_precipitation_de_mm', 'solar_gw']

In [ ]:
X = df_m_all_hr[input_features_m_init].iloc[:pred_start_point]
y = df_m_all_hr[target_feature_m].iloc[:pred_start_point]

X_for_pred = df_m_all_hr[input_features_m_init].iloc[pred_start_point:]
y_for_pred = df_m_all_hr[target_feature_m].iloc[pred_start_point:]

print("Training and test sets has {} samples in total.".format(X.shape[0]))
print("Prediction set has {} samples.".format(X_for_pred.shape[0]))


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=29*1.0/(total_rows-pred_rows), random_state=RSEED)

# Show the results of the split
print("Training set has {} samples.".format(X_train.shape[0]))
print("Testing set has {} samples.".format(X_test.shape[0]))
print(f"y_train: {y_train.shape}")
print(f"y_test: {y_test.shape}")

### XGBoost (unscaled) forecasting

In [ ]:
# Randomized search (run time: ~3m)

param_grid = {
    "subsample": list(np.arange(0.1, 0.7, 0.05)),
    "reg_lambda": [1e-5, 1e-2, 0.1, 0.5, 1, 2, 5, 10, 100],
    "reg_alpha": [1e-5, 1e-2, 0.1, 1, 5, 10, 20, 100],
    "n_estimators": list(np.arange(10, 200, 10)), # default 100
    "max_depth": list(np.arange(2, 13, 1)), # default 3
    "learning_rate": [0.0001, 0.0003, 0.001, 0.003, 0.01, 0.03, 0.05, 0.1, 0.15, 0.2, 0,3], # default 0.1
    "gamma": list(np.arange(0, 0.6, 0.05)),
    "colsample_bytree": list(np.arange(0.3, 1.0, 0.05))    
}

total_iters = 20000
cv_folds = 5

# Estimator for use in random search
estimator = XGBRegressor(random_state = RSEED)

# Create the random search model
xgb_rs_p = RandomizedSearchCV(estimator, param_grid, n_jobs = -1, 
                        scoring = 'r2', cv = cv_folds, 
                        n_iter = total_iters, verbose = 0, random_state=RSEED)

# Fit 
xgb_rs_p.fit(X_train, y_train)

In [ ]:
print('Best score:\n{:.2f}'.format(xgb_rs_p.best_score_))
print("Best parameters:\n{}".format(xgb_rs_p.best_params_))

In [ ]:
# Evaluate different models by comparing r2_score, RMSE, and MAE

# Function to calculate RMSE and r2_score
def cal_scores_triple(y_test, y_pred, y_train, y_train_pred, y_for_pred, y_for_pred_pred, clf_method_name='xgboost'): 
    
    print("====="*10)
    print(f"'{clf_method_name}'")
    print("-----"*2)
    
    print(f"          {'r2_score':<12}{'RMSE':<12}{'MAE':<12}")
    print(f"Train:    {r2_score(y_train,y_train_pred):<12.3f}{mean_squared_error(y_train,y_train_pred,squared=False):<12.3f}{mean_absolute_error(y_train,y_train_pred):<12.3f}")
    print(f"Test:     {r2_score(y_test,y_pred):<12.3f}{mean_squared_error(y_test,y_pred,squared=False):<12.3f}{mean_absolute_error(y_test,y_pred):<12.3f}")
    print(f"Forecast: {r2_score(y_for_pred,y_for_pred_pred):<12.3f}{mean_squared_error(y_for_pred,y_for_pred_pred,squared=False):<12.3f}{mean_absolute_error(y_for_pred, y_for_pred_pred):<12.3f}")

In [ ]:
# Apply best model for RS XGBoost

best_model_xgbrs_p = xgb_rs_p.best_estimator_

y_pred_xgbrs_p_best = best_model_xgbrs_p.predict(X_test)
y_train_pred_xgbrs_p_best = best_model_xgbrs_p.predict(X_train)
y_for_pred_pred_xgbrs_p_best = best_model_xgbrs_p.predict(X_for_pred)

cal_scores_triple(y_test, y_pred_xgbrs_p_best, y_train, y_train_pred_xgbrs_p_best, y_for_pred, y_for_pred_pred_xgbrs_p_best, 'Randomized Search XGBoost: best scores')

In [ ]:
# Plotting test values

# Draw a scatter plot with regression line
plt.figure(figsize=(6, 4))
plt.scatter(y_test, y_pred_xgbrs_p_best, alpha=0.7)
plt.plot(np.linspace(np.min(y_test), np.max(y_test), 100), np.linspace(np.min(y_test), np.max(y_test), 100), color='red', linestyle='--')
plt.xlabel("Actual Target Values")
plt.ylabel("Predicted Target Values")
plt.title("Predicted values vs. y_test dataset")
plt.grid(True)
plt.show()

In [ ]:
# Plotting forecast values

# Draw a scatter plot with regression line
plt.figure(figsize=(6, 4))
plt.scatter(y_for_pred, y_for_pred_pred_xgbrs_p_best, alpha=0.7)
plt.plot(np.linspace(np.min(y_for_pred), np.max(y_for_pred), 100), np.linspace(np.min(y_for_pred), np.max(y_for_pred), 100), color='red', linestyle='--')
plt.xlabel("Actual Target Values")
plt.ylabel("Predicted Target Values")
plt.title("Predicted values vs. y_for_pred dataset")
plt.grid(True)
plt.show()

### Ridge forecasting

In [ ]:
# Scale the input features using StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_for_pred_scaled = scaler.transform(X_for_pred)

In [ ]:
# Create the RidgeCV model and specify a list of candidate alpha values

alphas = list(np.arange(0.01, 0.21, 0.01)) + [2, 5, 10, 20, 30, 50, 100, 150, 200, 500, 1000, 2000]

# Create a list to store R2 scores for each alpha
r2_scores = []

# Create the Ridge model and perform cross-validation for each alpha value
for alpha in alphas:
    ridge_model = Ridge(alpha=alpha)
    # Perform cross-validation and calculate the R2 score for each fold
    cv_r2_scores = cross_val_score(ridge_model, X_train_scaled, y_train, cv=5, scoring='r2')
    # Calculate the mean R2 score across all folds for this alpha
    mean_r2_score = np.mean(cv_r2_scores)
    # Store the mean R2 score for this alpha in the list
    r2_scores.append(mean_r2_score)
    print(f"Alpha: {alpha}, Mean R2 Score: {mean_r2_score:.4f}")

# Find the best alpha value based on the highest R2 score
best_alpha = alphas[np.argmax(r2_scores)]

# Print the best alpha and its corresponding R2 score
print("\n")
print("====="*10)
print("'Alpha tuning'")
print("-----"*3)
print("Best Alpha:", best_alpha)
print("R2 Score (best Alpha):", max(r2_scores))

# Train the final Ridge model using the best alpha
ridge_model = Ridge(alpha=best_alpha)

# Fit the model to the training data
ridge_model.fit(X_train_scaled, y_train)

# Make predictions on the test data and for-pred data using the tuned Ridge model
y_pred = ridge_model.predict(X_test_scaled)
y_train_pred = ridge_model.predict(X_train_scaled)
y_for_pred_pred = ridge_model.predict(X_for_pred_scaled)

# Calculate MSE, RMSE, and R2 on the train data
mse_tr = mean_squared_error(y_train, y_train_pred)
rmse_tr = mean_squared_error(y_train, y_train_pred, squared=False)
r2_tr = r2_score(y_train, y_train_pred)

print("====="*10)
print("'Training'")
print("-----"*3)
print("MSE (test):", mse_tr)
print("RMSE (test):", rmse_tr)
print("R2 Score (test):", r2_tr)

# Calculate MSE, RMSE, and R2 on the test data
mse_t = mean_squared_error(y_test, y_pred)
rmse_t = mean_squared_error(y_test, y_pred, squared=False)
r2_t = r2_score(y_test, y_pred)

print("====="*10)
print("'Testing'")
print("-----"*3)
print("MSE (test):", mse_t)
print("RMSE (test):", rmse_t)
print("R2 Score (test):", r2_t)

# Calculate MSE, RMSE, and R2 on the for-pred data
mse_p = mean_squared_error(y_for_pred, y_for_pred_pred)
rmse_p = mean_squared_error(y_for_pred, y_for_pred_pred, squared=False)
r2_p = r2_score(y_for_pred, y_for_pred_pred)

print("====="*10)
print("'Forecasting'")
print("-----"*3)
print("MSE (forecast):", mse_p)
print("RMSE (forecast):", rmse_p)
print("R2 Score (forecast):", r2_p)
print("====="*10)

# Get the scaled coefficients and intercept of the model
coefficients = ridge_model.coef_
intercept = ridge_model.intercept_

# Transform the coefficients back to the original scale
original_scale_coefficients = coefficients / scaler.scale_

# Transform the intercept back to the original scale
original_scale_intercept = intercept - np.sum(coefficients * scaler.mean_ / scaler.scale_)


print(f"Scaled coefficients: {coefficients}")
print(f"scaler.scale_: {scaler.scale_}")
print(f"Scaled intercept: {intercept}")
print("Now transform the coefficients and intercept back to the original scale...")

# Convert the coefficients to a nested Python list using tolist()
coefficients_list = original_scale_coefficients.tolist()

# Use list comprehension to extract the floating-point values from the nested list
coefficients_float_list = [coef for coef in coefficients_list[0]]

print("\nRidge Regression Coefficients (original scale):")
for feature, coef in zip(X_train.columns, coefficients_float_list):
    print(f"{feature}: {coef:.6f}")

# Display the exact fitting function
print("\nExact Fitting Function:")
print(f"y = {original_scale_intercept[0]:.4f}", end="")
for i, coef in enumerate(coefficients_float_list):
    print(f" + {coef:.4f} * x{i+1}", end="")
print()

In [ ]:
# Plotting test values

# Draw a scatter plot with regression line
plt.figure(figsize=(6, 4))
plt.scatter(y_test, y_pred, alpha=0.7)
plt.plot(np.linspace(np.min(y_test), np.max(y_test), 100), np.linspace(np.min(y_test), np.max(y_test), 100), color='red', linestyle='--')
plt.xlabel("Actual Target Values")
plt.ylabel("Predicted Target Values")
plt.title("Predicted values vs. y_test dataset")
plt.grid(True)
plt.show()

In [ ]:
# Plotting forecast values

# Draw a scatter plot with regression line
plt.figure(figsize=(6, 4))
plt.scatter(y_for_pred, y_for_pred_pred, alpha=0.7)
plt.plot(np.linspace(np.min(y_for_pred), np.max(y_for_pred), 100), np.linspace(np.min(y_for_pred), np.max(y_for_pred), 100), color='red', linestyle='--')
plt.xlabel("Actual Target Values")
plt.ylabel("Predicted Target Values")
plt.title("Predicted values vs. y_for_pred dataset")
plt.grid(True)
plt.show()

In [ ]:
# Assemble DataFrames for plotting pred. vs. actual

# Assemble DataFrame for y_test and y_pred
y_test_values = y_test['solar_twh'].values

# Convert 'y_pred' to a pandas Series
y_test_col = pd.Series(y_test_values, index=y_test.index)
y_pred_col = pd.Series([val[0] for val in y_pred], index=y_test.index)

# Create a new DataFrame
df_test_pred = pd.DataFrame({'y_test_col': y_test_col, 'y_pred_col': y_pred_col})



# Assemble DataFrame for y_train and y_train_pred
y_train_values = y_train['solar_twh'].values

# Convert 'y_train_pred' to a pandas Series
y_train_col = pd.Series(y_train_values, index=y_train.index)
y_train_pred_col = pd.Series([val[0] for val in y_train_pred], index=y_train.index)

# Create a new DataFrame
df_train_pred = pd.DataFrame({'y_train_col': y_train_col, 'y_train_pred_col': y_train_pred_col})



# Assemble DataFrame for y_for_pred and y_for_pred_pred
y_for_pred_values = y_for_pred['solar_twh'].values

# Convert 'y_for_pred_pred' to a pandas Series
y_for_pred_col = pd.Series(y_for_pred_values, index=y_for_pred.index)
y_for_pred_pred_col = pd.Series([val[0] for val in y_for_pred_pred], index=y_for_pred.index)

# Create a new DataFrame
df_for_pred_pred = pd.DataFrame({'y_for_pred_col': y_for_pred_col, 'y_for_pred_pred_col': y_for_pred_pred_col})



print(df_for_pred_pred)

In [ ]:
# Concatenate DataFrames based on matching index
df_m_all_pred_plot = pd.concat([df_m_all, df_train_pred, df_test_pred, df_for_pred_pred], axis=1)

df_m_all_pred_plot.head(110)

In [ ]:
# Draw the scatter plot

# Define the columns to be plotted
columns_to_plot = ['y_train_col', 'y_train_pred_col', 'y_test_col', 'y_pred_col', 'y_for_pred_col', 'y_for_pred_pred_col']

# Define the colors for each column
colors = ['blue', 'green', 'red', 'orange', 'purple', 'brown']

# Plot each column using a scatter plot with a different color
plt.figure(figsize=(12, 6))
for col, color in zip(columns_to_plot, colors):
    plt.scatter(df_m_all_pred_plot['dt_date'], df_m_all_pred_plot[col], color=color, label=col)

# Set the x-axis label and rotate the x-axis tick labels for better visibility
plt.xlabel('Date')
plt.xticks(rotation=45)

# Set the y-axis label
plt.ylabel('Values')

# Add a legend to the plot to identify the columns
plt.legend()

# Show the plot
plt.show()


In [ ]:
# Scatter plot overlaid with line plot

# Convert the 'dt_date' column to datetime format
df_m_all_pred_plot['dt_date'] = pd.to_datetime(df_m_all_pred_plot['dt_date'], format='%Y-%m')

# Define the columns for each group
group_actu_columns = ['y_train_col', 'y_test_col', 'y_for_pred_col']
group_pred_columns = ['y_train_pred_col', 'y_pred_col', 'y_for_pred_pred_col']

group1_names = ['y_train: dataset', 'y_test: dataset', 'y_for_forecast: dataset']
group2_names = ['y_train: predictions', 'y_test: predictions', 'y_for_forecast: predictions']

# Define the colors for each column
colors = ['blue', 'red', 'purple', 'green', 'orange', 'brown']

# Create the plot with scatter plots for each column in both groups
plt.figure(figsize=(12, 6))

for i, col in enumerate(group_actu_columns + group_pred_columns):
    plt.scatter(df_m_all_pred_plot['dt_date'], df_m_all_pred_plot[col], color=colors[i], label=col)

# Combine the values of the three columns in each group into a single column
df_m_all_pred_plot['group1_values'] = df_m_all_pred_plot[group_actu_columns].bfill(axis=1).iloc[:, 0]
df_m_all_pred_plot['group2_values'] = df_m_all_pred_plot[group_pred_columns].bfill(axis=1).iloc[:, 0]

# Sort the DataFrame by 'dt_date' for proper linking of dots in each group
df_m_all_pred_plot = df_m_all_pred_plot.sort_values('dt_date')

# Connect the dots with lines for each group
plt.plot(df_m_all_pred_plot['dt_date'], df_m_all_pred_plot['group1_values'], color='black', linestyle='-', linewidth=1, label='Actual data')
plt.plot(df_m_all_pred_plot['dt_date'], df_m_all_pred_plot['group2_values'], color='magenta', linestyle='--', linewidth=1, label='Predictions')

# Set labels, legend, and show the plot
plt.xlabel('Date')
plt.xticks(rotation=0)
plt.ylabel("Germany's monthly solar energy generation [TWh]")
plt.legend()

# Retrieve the legend object
legend = plt.legend()

# Modify the legend labels
for i, label in enumerate(legend.get_texts()):
    if i < len(group1_names):
        # Modify the labels for Group 1 columns
        label.set_text(group1_names[i])
    elif i < len(group1_names)+len(group2_names):
        # Modify the labels for Group 2 columns
        label.set_text(group2_names[i - len(group1_names)])
    else:
        break

plt.show()


In [ ]:
# Scatter plot overlaid with line plot: alternative way of changing legend labels

# Convert the 'dt_date' column to datetime format
df_m_all_pred_plot['dt_date'] = pd.to_datetime(df_m_all_pred_plot['dt_date'], format='%Y-%m')

# Define the columns for each group and their corresponding display names
group1_columns = ['y_train_col', 'y_test_col', 'y_for_pred_col']
group2_columns = ['y_train_pred_col', 'y_pred_col', 'y_for_pred_pred_col']

group1_names = ['y_train: dataset', 'y_test: dataset', 'y_for_forecast: dataset']
group2_names = ['y_train: predictions', 'y_test: predictions', 'y_for_forecast: predictions']

# Define the colors for each column
colors = itertools.cycle(['blue', 'red', 'purple', 'green', 'orange', 'brown'])

# Create the plot with scatter plots for each column in both groups
plt.figure(figsize=(12, 6))

# Initialize empty lists to store the unique columns and their corresponding names
unique_columns = []
unique_names = []

for i, col in enumerate(group1_columns + group2_columns):
    # Check if the column is already in the unique_columns list
    if col not in unique_columns:
        unique_columns.append(col)
        if col in group1_columns:
            # If it's a group 1 column, get its corresponding name from group1_names
            unique_names.append(group1_names[group1_columns.index(col)])
        else:
            # If it's a group 2 column, get its corresponding name from group2_names
            unique_names.append(group2_names[group2_columns.index(col)])

    color = next(colors)
    plt.scatter(df_m_all_pred_plot['dt_date'], df_m_all_pred_plot[col], color=color, label=unique_names[-1])

# Combine the values of the three columns in each group into a single column
df_m_all_pred_plot['group1_values'] = df_m_all_pred_plot[group1_columns].bfill(axis=1).iloc[:, 0]
df_m_all_pred_plot['group2_values'] = df_m_all_pred_plot[group2_columns].bfill(axis=1).iloc[:, 0]

# Sort the DataFrame by 'dt_date' for proper linking of dots in each group
df_m_all_pred_plot = df_m_all_pred_plot.sort_values('dt_date')

# Connect the dots with lines for each group
plt.plot(df_m_all_pred_plot['dt_date'], df_m_all_pred_plot['group1_values'], color='black', linestyle='-', linewidth=1, label='Actual data')
plt.plot(df_m_all_pred_plot['dt_date'], df_m_all_pred_plot['group2_values'], color='magenta', linestyle='--', linewidth=1, label='Predictions')

# Set labels and legend
plt.xlabel('Date')
plt.xticks(rotation=0)
plt.ylabel("Germany's monthly solar energy generation [TWh]")

# Show the plot
plt.legend()
# plt.tight_layout()
plt.show()


In [ ]:
# Scatter plot overlaid with line plot: tune dot size, transparency, and font size

# Convert the 'dt_date' column to datetime format
df_m_all_pred_plot['dt_date'] = pd.to_datetime(df_m_all_pred_plot['dt_date'], format='%Y-%m')

# Define the columns for each group and their corresponding display names
group1_columns = ['y_train_col', 'y_test_col', 'y_for_pred_col']
group2_columns = ['y_train_pred_col', 'y_pred_col', 'y_for_pred_pred_col']

group1_names = ['Model training: dataset', 'Model testing: dataset', 'Model forecasting: dataset']
group2_names = ['Model training: predictions', 'Model testing: predictions', 'Model forecasting: predictions']

# Define the colors for each column
# colors = itertools.cycle(['blue', 'red', 'purple', 'green', 'orange', 'brown'])
colors = itertools.cycle(['green', 'blue', 'red'])
# mpl.rcParams['font.size'] = 16

# Create the plot with scatter plots for each column in both groups
plt.figure(figsize=(12, 6))

# Initialize empty lists to store the unique columns and their corresponding names
unique_columns = []
unique_names = []

for i, col in enumerate(group1_columns + group2_columns):
    # Check if the column is already in the unique_columns list
    if col not in unique_columns:
        unique_columns.append(col)
        if col in group1_columns:
            # If it's a group 1 column, get its corresponding name from group1_names
            unique_names.append(group1_names[group1_columns.index(col)])
        else:
            # If it's a group 2 column, get its corresponding name from group2_names
            unique_names.append(group2_names[group2_columns.index(col)])

    color = next(colors)
    if col in group1_columns:
        # If it's a group 1 column, use half-transparent dots with reduced size
        plt.scatter(df_m_all_pred_plot['dt_date'], df_m_all_pred_plot[col], color=color, label=unique_names[-1], alpha=0.5, s=30)
    else:
        # If it's a group 2 column, use empty circles for the dots
        plt.scatter(df_m_all_pred_plot['dt_date'], df_m_all_pred_plot[col], color='none', edgecolors=color, label=unique_names[-1], s=30)

# Combine the values of the three columns in each group into a single column
df_m_all_pred_plot['group1_values'] = df_m_all_pred_plot[group1_columns].bfill(axis=1).iloc[:, 0]
df_m_all_pred_plot['group2_values'] = df_m_all_pred_plot[group2_columns].bfill(axis=1).iloc[:, 0]

# Sort the DataFrame by 'dt_date' for proper linking of dots in each group
df_m_all_pred_plot = df_m_all_pred_plot.sort_values('dt_date')

# Connect the dots with lines for each group
plt.plot(df_m_all_pred_plot['dt_date'], df_m_all_pred_plot['group1_values'], color='black', linestyle='-', linewidth=1, label='Actual data')
plt.plot(df_m_all_pred_plot['dt_date'], df_m_all_pred_plot['group2_values'], color='brown', linestyle='--', linewidth=1, label='Predictions')

# Set labels and legend
plt.xlabel('Date', fontsize=12)
plt.xticks(rotation=0, fontsize=12)
plt.ylabel("Solar energy generation [TWh]", fontsize=12)
plt.yticks(fontsize=12)
plt.title("Germany's monthly solar energy generation: actual vs. modeled", fontsize=14)

# Show the plot
# plt.legend()
plt.legend(fontsize=10)
# plt.tight_layout()
plt.show()


In [ ]:
df_daily.head(2)

In [ ]:
df_daily_idx.head(2)

In [ ]:
# # Export 'df_daily' to CSV
# df_daily.to_csv('../output/df_daily.csv', sep=',', index=False)

In [ ]:
# Export 'df_m_all' to CSV
# df_m_all.to_csv('../output/df_m_all.csv', sep=',', index=False)

df_m_all.head(2)